In [ ]:
import geopandas as gpd
import pandas as pd
lia = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/Himalaya_LIA_with_RGIID_Complete/Himalaya_LIA_with_RGIID_Complete.shp")
rgi = gpd.read_file("/mnt/e/2Data/022_工作2/出图数据/himalaya_glaciers_complete/himalaya_glaciers_complete.shp")

In [ ]:
# 筛选 RGI 面积 >= 10 km²
rgi_large = rgi[rgi['Area'] >= 10].copy()
large_rgi_ids = set(rgi_large['RGIId'])

# 找包含这些大冰川的 LIA
def contains_large_rgi(rgi_str):
    if pd.isna(rgi_str):
        return False
    ids = [x.strip() for x in str(rgi_str).split(';')]
    return bool(set(ids) & large_rgi_ids)

lia_matched = lia[lia['AllRGIs'].apply(contains_large_rgi)].copy()

print("LIA筛选完成: {} 条".format(len(lia_matched)))

In [ ]:
# 筛选MainRGI对应的RGI冰川
main_rgi_ids = lia_matched['MainRGI'].tolist()
rgi_to_calibrate = rgi[rgi['RGIId'].isin(main_rgi_ids)].copy()

# 转WGS84
lia_matched = lia_matched.to_crs("EPSG:4326")
rgi_to_calibrate = rgi_to_calibrate.to_crs("EPSG:4326")

# 建立映射字典: LIA的UniqueID → MainRGI
lia_to_main_rgi = dict(zip(
    lia_matched['UniqueID'].astype(int),
    lia_matched['MainRGI']
))

print("RGI待校准: {} 条".format(len(rgi_to_calibrate)))
print("映射关系: {} 条".format(len(lia_to_main_rgi)))

In [ ]:
from oggm import cfg, workflow, tasks, utils

WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_rgi_calibration"
DEM_PATH = "/mnt/d/01 文件整理/04 数据/HMA_newregion/DEM_clip/Himalaya_3regions_DEM.tif"
MB_PATHS = [
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_13/dh_13_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_14/dh_14_rgi60_pergla_rates.csv",
    "/mnt/d/01 文件整理/04 数据/03 物质平衡数据/hugonnet time_series_15/dh_15_rgi60_pergla_rates.csv",
]
MB_PERIOD = "2000-01-01_2020-01-01"
# 读取MB数据
dfs = []
for path in MB_PATHS:
    print("读取: {}".format(path))
    df = pd.read_csv(path)
    dfs.append(df)
mb_all = pd.concat(dfs, ignore_index=True)
mb_data = mb_all[mb_all['period'] == MB_PERIOD].copy()
mb_data['dmdtda_kg'] = mb_data['dmdtda'] * 1000
mb_data = mb_data.set_index('rgiid')
print("MB数据: {} 条".format(len(mb_data)))
# 清理RGI数据
rgi = rgi_to_calibrate.copy()
rgi = rgi[rgi.geometry.notna() & ~rgi.is_empty].copy()
rgi['geometry'] = rgi.geometry.buffer(0)
rgi = rgi[rgi.is_valid].copy()
rgi = rgi[rgi['RGIId'].isin(mb_data.index)].copy()
print("有MB数据的RGI冰川: {}".format(len(rgi)))

In [ ]:
from oggm import cfg, workflow, tasks, GlacierDirectory
cfg.initialize()

cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

gdirs_rgi = workflow.init_glacier_directories(rgi, reset=False)
print("初始化完成: {}".format(len(gdirs_rgi)))

In [ ]:
for gdir in gdirs_rgi:
    tasks.define_glacier_region(gdir, source='USER')
print("define_glacier_region完成: {}".format(len(gdirs_rgi)))

In [ ]:
cfg.PARAMS['baseline_climate'] = 'CRU'
for gdir in gdirs_rgi:
    tasks.process_cru_data(gdir)
print("气候数据处理完成")

In [ ]:
from oggm.workflow import execute_entity_task
cfg.PARAMS['use_multiprocessing'] = True

execute_entity_task(tasks.glacier_masks, gdirs_rgi)
execute_entity_task(tasks.compute_centerlines, gdirs_rgi)
execute_entity_task(tasks.initialize_flowlines, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_line, gdirs_rgi)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs_rgi)
execute_entity_task(tasks.catchment_area, gdirs_rgi)
execute_entity_task(tasks.catchment_intersections, gdirs_rgi)
execute_entity_task(tasks.catchment_width_geom, gdirs_rgi)
execute_entity_task(tasks.catchment_width_correction, gdirs_rgi)
print("预处理完成")

In [ ]:
cfg.PARAMS['use_winter_prcp_fac'] = False
cfg.PARAMS['prcp_fac'] = 8
from oggm.core.massbalance import mb_calibration_from_scalar_mb
import os
import json
success = 0
fail = 0
for gdir in gdirs_rgi:
    try:
        ref_mb = float(mb_data.loc[gdir.rgi_id, 'dmdtda_kg'])
        mb_calibration_from_scalar_mb(
            gdir,
            ref_mb=ref_mb,
            ref_mb_years=(2000, 2020),
            overwrite_gdir=True,
            calibrate_param1='melt_f',
            calibrate_param2='temp_bias',
        )
        success += 1
    except Exception as e:
        fail += 1
        if fail <= 3:
            print("失败: {} - {}".format(gdir.rgi_id, str(e)[:100]))

print("\nCRU校准完成: 成功 {}, 失败 {}".format(success, fail))

# 验证几条
for gdir in gdirs_rgi[:5]:
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        p = json.load(f)
    print("{}: melt_f={:.3f} temp_bias={:.3f} baseline={}".format(
        gdir.rgi_id, p['melt_f'], p['temp_bias'], p['baseline_climate_source']))

In [ ]:
lia_sorted = lia_matched.sort_values('UniqueID').copy()
lia_cooked = utils.cook_rgidf(
    lia_sorted,
    o1_region='15',
    bgndate='16500101',
    ids=lia_sorted['UniqueID'].astype(int).tolist(),
    assign_column_values={'LIAArea': 'Area'}
)

lia_cooked['UniqueID'] = lia_sorted['UniqueID'].values
lia_cooked['MainRGI'] = lia_sorted['MainRGI'].values

print("cook完成: {} 条".format(len(lia_cooked)))
print(lia_cooked[['RGIId', 'UniqueID', 'Area']].head())

In [ ]:
LIA_WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_oggm_LIA_spinup"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"


cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20

gdirs_lia = workflow.init_glacier_directories(lia_cooked,reset= False)
print("LIA初始化完成: {} 条".format(len(gdirs_lia)))

In [ ]:
# 1. 重建lia_mb_params（用新的CRU校准参数）
import json, os, numpy as np

rgi_params_cru = {}
for gdir in gdirs_rgi:
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        rgi_params_cru[gdir.rgi_id] = json.load(f)

lia_mb_params_cru = {}
missing = []
for idx, row in lia_matched.iterrows():
    unique_id = int(row['UniqueID'])
    main_rgi = row['MainRGI']
    if main_rgi in rgi_params_cru:
        p = rgi_params_cru[main_rgi]
        lia_mb_params_cru[unique_id] = {
            'melt_f': p['melt_f'],
            'prcp_fac': p['prcp_fac'],
            'temp_bias': p['temp_bias'],
        }
    else:
        missing.append((unique_id, main_rgi))

# 缺失的用区域平均
avg_melt_f = np.mean([p['melt_f'] for p in rgi_params_cru.values()])
avg_temp_bias = np.mean([p['temp_bias'] for p in rgi_params_cru.values()])
for uid, rgi in missing:
    lia_mb_params_cru[uid] = {'melt_f': avg_melt_f, 'prcp_fac': 8.0, 'temp_bias': avg_temp_bias}

print("LIA参数映射: {} 条, 缺失补充: {} 条".format(len(lia_mb_params_cru), len(missing)))
print("melt_f范围: {:.2f} ~ {:.2f}".format(
    min(p['melt_f'] for p in lia_mb_params_cru.values()),
    max(p['melt_f'] for p in lia_mb_params_cru.values())))

# 2. 写入LIA冰川的mb_calib.json
for gdir in gdirs_lia:
    unique_id = int(gdir.rgi_id.split('.')[-1])
    params = lia_mb_params_cru[unique_id]

    mb_calib = {
        'rgi_id': gdir.rgi_id,
        'bias': 0,
        'melt_f': params['melt_f'],
        'prcp_fac': params['prcp_fac'],
        'temp_bias': params['temp_bias'],
        'reference_mb': 0,
        'reference_mb_err': None,
        'reference_period': '2000-01-01_2020-01-01',
        'mb_global_params': {
            'temp_default_gradient': -0.0065,
            'temp_all_solid': 0.0,
            'temp_all_liq': 2.0,
            'temp_melt': -1.0,
        },
        'baseline_climate_source': 'CRU TS4.04',
    }

    fpath = os.path.join(gdir.dir, 'mb_calib.json')
    with open(fpath, 'w') as f:
        json.dump(mb_calib, f, indent=2)

print("LIA冰川mb_calib写入完成: {} 条".format(len(gdirs_lia)))

# 验证IK
with open(os.path.join(gdirs_lia[0].dir, 'mb_calib.json'), 'r') as f:
    p = json.load(f)
print("样本: melt_f={:.3f} baseline={}".format(p['melt_f'], p['baseline_climate_source']))

In [ ]:
for gdir in gdirs_lia:
    tasks.define_glacier_region(gdir, source='USER')
print("define_glacier_region完成: {}".format(len(gdirs_lia)))

In [ ]:
from oggm.workflow import execute_entity_task

execute_entity_task(tasks.glacier_masks, gdirs_lia)
execute_entity_task(tasks.compute_centerlines, gdirs_lia)
execute_entity_task(tasks.initialize_flowlines, gdirs_lia)
execute_entity_task(tasks.compute_downstream_line, gdirs_lia)
execute_entity_task(tasks.compute_downstream_bedshape, gdirs_lia)
execute_entity_task(tasks.catchment_area, gdirs_lia)
execute_entity_task(tasks.catchment_intersections, gdirs_lia)
execute_entity_task(tasks.catchment_width_geom, gdirs_lia)
execute_entity_task(tasks.catchment_width_correction, gdirs_lia)
print("预处理完成: {}".format(len(gdirs_lia)))

In [ ]:
from tqdm import tqdm

os.environ['http_proxy'] = 'http://192.168.160.1:7890'
os.environ['https_proxy'] = 'http://192.168.160.1:7890'


# cfg.PARAMS['baseline_climate'] = 'CRU'
LIA_WORKING_DIR = "/mnt/f/oggm_related/oggm_LIA_simulation1"
BARE_DEM_PATH = "/mnt/f/oggm_related/oggm_LIA_himalaya_great/output/Himalaya_bare_DEM.tif"


cfg.PATHS['working_dir'] = LIA_WORKING_DIR
cfg.PATHS['dem_file'] = BARE_DEM_PATH
cfg.PARAMS['border'] = 360
cfg.PARAMS['use_intersects'] = False
cfg.PARAMS['use_rgi_area'] = False
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 20
#
cfg.PARAMS['baseline_climate'] = 'CRU'
for gdir in gdirs_lia:
    tasks.process_cru_data(gdir)


In [ ]:
# 处理重建的气候数据
import xarray as xr
import os
from oggm.shop import gcm_climate

ensemble_dir = "/mnt/e/3spyderProject/HMA_500/Down_scale_new/out_data/oggm_climate/Ensemble"
climate_files = sorted([f for f in os.listdir(ensemble_dir) if f.endswith('.nc')])

for ci, cfile in enumerate(climate_files):
    suffix = '_' + cfile.replace('climate_oggm_', '').replace('.nc', '')
    fpath = os.path.join(ensemble_dir, cfile)

    print("\n[{}/{}] {} -> suffix: {}".format(ci+1, len(climate_files), cfile, suffix))

    ds_clim = xr.open_dataset(fpath, use_cftime=True)
    temp_k = ds_clim['temp'] + 273.15
    prcp = ds_clim['prcp']

    success = 0
    fail = 0

    for gdir in gdirs_lia:
        try:
            temp_sel = temp_k.sel(lon=gdir.cenlon, lat=gdir.cenlat, method='nearest')
            prcp_sel = prcp.sel(lon=gdir.cenlon, lat=gdir.cenlat, method='nearest')

            gcm_climate.process_gcm_data(
                gdir,
                temp=temp_sel,
                prcp=prcp_sel,
                year_range=('1961', '1990'),
                source='rec' + suffix,
                output_filesuffix=suffix,
            )
            success += 1
        except Exception as e:
            fail += 1
            if fail <= 3:
                print("  失败: {} - {}".format(gdir.rgi_id, str(e)[:60]))

    ds_clim.close()
    print("  完成: 成功 {}, 失败 {}".format(success, fail))

print("\n全部处理完成!")

In [ ]:
import json
import os

mb_global_params = {
    'temp_default_gradient': -0.0065,
    'temp_all_solid': 0.0,
    'temp_all_liq': 2.0,
    'temp_melt': -1.0
}

success = 0
for gdir in gdirs_lia:
    try:
        unique_id = int(gdir.rgi_id.split('.')[-1])
        params = lia_mb_params_cru[unique_id]

        mb_calib = {
            'rgi_id': gdir.rgi_id,
            'melt_f': params['melt_f'],
            'prcp_fac': params['prcp_fac'],
            'temp_bias': params['temp_bias'],
            'bias': 0,
            'reference_mb': 0,
            'reference_mb_err': None,
            'reference_period': '2000-01-01_2020-01-01',
            'mb_global_params': mb_global_params,
            'baseline_climate_source': 'CRU TS4.04',
        }

        fpath = os.path.join(gdir.dir, 'mb_calib.json')
        with open(fpath, 'w') as f:
            json.dump(mb_calib, f, indent=2)
        success += 1
    except Exception as e:
        print("失败: {} - {}".format(gdir.rgi_id, str(e)[:60]))

print("mb_calib更新完成: {}".format(success))


# 2. 验证一下climate_historical和mb_calib的source是否匹配
import xarray as xr
gdir = gdirs_lia[0]
ds = xr.open_dataset(os.path.join(gdir.dir, 'climate_historical.nc'))
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
    mb = json.load(f)
print("\nclimate_historical source: {}".format(ds.attrs.get('climate_source')))
print("mb_calib source: {}".format(mb['baseline_climate_source']))
ds.close()

In [ ]:
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task
from oggm import tasks, cfg
import json, os, numpy as np, xarray as xr
from scipy.interpolate import interp1d
import pandas as pd
import pickle

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

windows = [1300, 1350, 1400, 1450, 1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850]
offsets = [0, -1, -2, -3, -4]

# 选一条冰川
gdir = gdirs_lia[0]
uid = int(gdir.rgi_id.split('.')[-1])
original_tb = lia_mb_params_cru[uid]['temp_bias']
print("测试冰川: {}, original temp_bias={:.2f}".format(gdir.rgi_id, original_tb))

# ========== 第一部分：运行所有窗口×offset ==========
for y0 in windows:
    for offset in offsets:
        with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
            p = json.load(f)
        p['temp_bias'] = original_tb + offset
        with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
            json.dump(p, f, indent=2)

        suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
        try:
            execute_entity_task(
                tasks.run_constant_climate,
                [gdir],
                nyears=300,
                y0=y0,
                halfsize=30,
                climate_filename='gcm_data',
                climate_input_filesuffix='_ensemble_rec_ref',
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                output_filesuffix=suffix,
                store_model_geometry=True,
                store_fl_diagnostics=True
            )
            ds = xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=suffix))
            pct = (ds['area_m2'].values[-1] - ds['area_m2'].values[0]) / ds['area_m2'].values[0] * 100
            print("y0={}, offset={}: area pct={:.1f}%".format(y0, offset, pct))
            ds.close()
        except Exception as e:
            print("y0={}, offset={}: FAILED - {}".format(y0, offset, e))

# 恢复temp_bias
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
    p = json.load(f)
p['temp_bias'] = original_tb
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
    json.dump(p, f, indent=2)
print("\n第一部分完成，temp_bias已恢复")

# ========== 第二部分：提取数据 ==========
save_dir = os.path.join(cfg.PATHS['working_dir'], 'experiment_results_single')
os.makedirs(save_dir, exist_ok=True)

# 2a. 保存时间序列（netCDF per window）
for y0 in windows:
    for offset in offsets:
        suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
        try:
            f = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
            ds = xr.open_dataset(f)
            ds_out = xr.Dataset({
                'area_m2': ('time', ds['area_m2'].values),
                'volume_m3': ('time', ds['volume_m3'].values),
                'length_m': ('time', ds['length_m'].values),
            }, coords={'time': np.arange(len(ds['area_m2'].values))})
            ds_out.attrs['rgi_id'] = gdir.rgi_id
            ds_out.attrs['y0'] = y0
            ds_out.attrs['offset'] = offset
            nc_path = os.path.join(save_dir, 'ts_w{}_offset{}.nc'.format(
                y0, str(offset).replace('.', '_').replace('-', 'n')))
            ds_out.to_netcdf(nc_path)
            ds.close()
        except:
            pass
    print("y0={} 时间序列保存完成".format(y0))

# 2b. 提取气候信息
climate_info = {}
for y0 in windows:
    try:
        fpath = gdir.get_filepath('gcm_data', filesuffix='_ensemble_rec_ref')
        ds_clim = xr.open_dataset(fpath)
        t_start = '{}-01'.format(y0 - 30)
        t_end = '{}-12'.format(y0 + 30)
        ds_sel = ds_clim.sel(time=slice(t_start, t_end))
        climate_info[y0] = {
            'mean_temp': float(ds_sel['temp'].mean()),
            'mean_prcp': float(ds_sel['prcp'].mean()),
        }
        ds_clim.close()
    except Exception as e:
        climate_info[y0] = {'mean_temp': np.nan, 'mean_prcp': np.nan, 'error': str(e)}
print("气候信息提取完成")

# 2c. 冰川基础属性
params = lia_mb_params_cru[uid]
try:
    suffix0 = '_w1650_offset0'
    ds = xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=suffix0))
    lia_area = float(ds['area_m2'].values[0])
    lia_volume = float(ds['volume_m3'].values[0])
    lia_length = float(ds['length_m'].values[0])
    ds.close()
except:
    lia_area = lia_volume = lia_length = np.nan

glacier_info = {
    gdir.rgi_id: {
        'melt_f': params['melt_f'],
        'temp_bias_original': params['temp_bias'],
        'prcp_fac': params['prcp_fac'],
        'lia_area_m2': lia_area,
        'lia_volume_m3': lia_volume,
        'lia_length_m': lia_length,
    }
}

# 2d. 计算best_offset
all_results = {}
for y0 in windows:
    area_changes = {}
    for offset in offsets:
        suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
        try:
            ds = xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=suffix))
            area_init = float(ds['area_m2'].values[0])
            area_end = float(ds['area_m2'].values[-1])
            pct = (area_end - area_init) / area_init * 100
            ds.close()
            area_changes[offset] = pct
        except:
            pass

    if len(area_changes) < 2:
        all_results[y0] = {'best_offset': np.nan, 'note': 'insufficient_data'}
        continue

    offs = sorted(area_changes.keys())
    pcts = [area_changes[o] for o in offs]

    best_offset = np.nan
    for i in range(len(pcts) - 1):
        if pcts[i] * pcts[i+1] <= 0:
            f_interp = interp1d([pcts[i], pcts[i+1]], [offs[i], offs[i+1]])
            best_offset = float(f_interp(0))
            break

    if np.isnan(best_offset):
        if all(p < 0 for p in pcts):
            note = 'all_shrink'
        elif all(p > 0 for p in pcts):
            note = 'all_grow'
        else:
            note = 'no_crossing'
        all_results[y0] = {'best_offset': np.nan, 'note': note}
    else:
        all_results[y0] = {'best_offset': best_offset, 'note': 'success'}
    print("y0={}: best_offset={}, note={}".format(
        y0, all_results[y0]['best_offset'], all_results[y0]['note']))

# ========== 第三部分：汇总CSV ==========
rows = []
rgi_id = gdir.rgi_id
gi = glacier_info[rgi_id]

for y0 in windows:
    for offset in offsets:
        suffix = '_w{}_offset{}'.format(y0, str(offset).replace('.', '_').replace('-', 'n'))
        try:
            ds = xr.open_dataset(gdir.get_filepath('model_diagnostics', filesuffix=suffix))
            row = {
                'y0': y0,
                'rgi_id': rgi_id,
                'offset': offset,
                'area_init': float(ds['area_m2'].values[0]),
                'area_end': float(ds['area_m2'].values[-1]),
                'area_pct': float((ds['area_m2'].values[-1] - ds['area_m2'].values[0]) / ds['area_m2'].values[0] * 100),
                'volume_init': float(ds['volume_m3'].values[0]),
                'volume_end': float(ds['volume_m3'].values[-1]),
                'length_init': float(ds['length_m'].values[0]),
                'length_end': float(ds['length_m'].values[-1]),
                'melt_f': gi['melt_f'],
                'temp_bias_original': gi['temp_bias_original'],
                'prcp_fac': gi['prcp_fac'],
                'lia_area_m2': gi['lia_area_m2'],
                'actual_temp_bias': gi['temp_bias_original'] + offset,
                'best_offset': all_results[y0]['best_offset'],
                'note': all_results[y0]['note'],
                'window_mean_temp': climate_info.get(y0, {}).get('mean_temp'),
                'window_mean_prcp': climate_info.get(y0, {}).get('mean_prcp'),
            }
            ds.close()
            rows.append(row)
        except:
            pass

df = pd.DataFrame(rows)
csv_path = os.path.join(save_dir, 'window_offset_single_{}.csv'.format(rgi_id.replace('.', '_')))
df.to_csv(csv_path, index=False)
print("\nCSV保存到: {}".format(csv_path))

# 保存pickle
pkl_path = os.path.join(save_dir, 'window_offset_single_{}.pkl'.format(rgi_id.replace('.', '_')))
with open(pkl_path, 'wb') as f:
    pickle.dump({
        'all_results': all_results,
        'glacier_info': glacier_info,
        'climate_info': climate_info,
        'windows': windows,
        'offsets': offsets,
        'rgi_id': rgi_id,
    }, f)
print("Pickle保存到: {}".format(pkl_path))
print("\n全部完成！{}".format(rgi_id))

In [ ]:
import xarray as xr
import numpy as np

fpath = gdir.get_filepath('gcm_data', filesuffix='_ensemble_rec_ref')
ds = xr.open_dataset(fpath, use_cftime=True)

temp = ds['temp'].values
time = ds['time'].values
years = np.array([t.year for t in time])

unique_years = np.unique(years)
temp_annual = []
for yr in unique_years:
    temp_annual.append(temp[years == yr].mean())
temp_annual = np.array(temp_annual)

# 参考期
ref_mask = (unique_years >= 1961) & (unique_years <= 1990)
t_ref = temp_annual[ref_mask].mean()

# 各时期距平
periods = [(1270, 1330), (1320, 1380), (1370, 1430), (1420, 1480),
           (1470, 1530), (1520, 1580), (1570, 1630), (1620, 1680),
           (1670, 1730), (1720, 1780), (1770, 1830), (1820, 1880),
           (1870, 1930), (1920, 1980), (1960, 2000)]

print("参考期(1961-1990)平均温度: {:.2f}°C".format(t_ref))
print("\n{:<16} {:>10} {:>10}".format("时期", "平均温度", "距平"))
print("-" * 38)
for y1, y2 in periods:
    mask = (unique_years >= y1) & (unique_years <= y2)
    if mask.sum() > 0:
        t_mean = temp_annual[mask].mean()
        print("{:<16} {:>10.2f} {:>+10.2f}".format("{}-{}".format(y1, y2), t_mean, t_mean - t_ref))

ds.close()

In [ ]:
import json, os
import xarray as xr
import numpy as np
import pickle
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task
from oggm import tasks
import pandas as pd

# 加载之前的结果
pkl_path = os.path.join(cfg.PATHS['working_dir'], 'experiment_results_single',
                        'window_offset_single_RGI60-15_00007.pkl')
with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

all_results = data['all_results']
windows = data['windows']

gdir = gdirs_lia[0]
uid = int(gdir.rgi_id.split('.')[-1])
original_tb = lia_mb_params_cru[uid]['temp_bias']
rgi_area = gdir.rgi_area_m2 / 1e6

print("冰川: {}, RGI观测面积: {:.2f} km2, original temp_bias: {:.2f}\n".format(
    gdir.rgi_id, rgi_area, original_tb))

results_rows = []

print("{:<8} {:>12} {:>12} {:>12} {:>10} {:>10}".format(
    "t0", "best_offset", "LIA_area", "2000_area", "change%", "sim/RGI"))
print("-" * 68)

for t0 in windows:
    bo = all_results[t0]['best_offset']
    if np.isnan(bo):
        print("{:<8} skipped (no best_offset)".format(t0))
        continue

    # 设置best_offset
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        p = json.load(f)
    p['temp_bias'] = original_tb + bo
    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
        json.dump(p, f, indent=2)

    # spin-up (带store_model_geometry)
    try:
        execute_entity_task(
            tasks.run_constant_climate,
            [gdir],
            nyears=300,
            y0=t0,
            halfsize=30,
            climate_filename='gcm_data',
            climate_input_filesuffix='_ensemble_rec_ref',
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            output_filesuffix='_spinup_t0_{}'.format(t0),
            store_model_geometry=True,
        )
    except Exception as e:
        print("{:<8} spinup FAILED: {}".format(t0, e))
        continue

    # 正演到2000
    try:
        execute_entity_task(
            tasks.run_from_climate_data,
            [gdir],
            min_ys=t0,
            max_ys=2000,
            climate_filename='gcm_data',
            climate_input_filesuffix='_ensemble_rec_ref',
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            init_model_filesuffix='_spinup_t0_{}'.format(t0),
            output_filesuffix='_forward_t0_{}'.format(t0),
        )
    except Exception as e:
        print("{:<8} forward FAILED: {}".format(t0, e))
        continue

    # 提取结果
    try:
        ds = xr.open_dataset(gdir.get_filepath('model_diagnostics',
                                                filesuffix='_forward_t0_{}'.format(t0)))
        a_init = float(ds['area_m2'].values[0]) / 1e6
        a_end = float(ds['area_m2'].values[-1]) / 1e6
        v_init = float(ds['volume_m3'].values[0])
        v_end = float(ds['volume_m3'].values[-1])
        l_init = float(ds['length_m'].values[0])
        l_end = float(ds['length_m'].values[-1])
        change = (a_end - a_init) / a_init * 100
        ratio = a_end / rgi_area
        ds.close()

        print("{:<8} {:>+12.3f} {:>12.2f} {:>12.2f} {:>+10.1f} {:>10.2f}".format(
            t0, bo, a_init, a_end, change, ratio))

        results_rows.append({
            't0': t0,
            'best_offset': bo,
            'actual_temp_bias': original_tb + bo,
            'lia_area_km2': a_init,
            'end_area_km2': a_end,
            'area_change_pct': change,
            'sim_rgi_ratio': ratio,
            'rgi_area_km2': rgi_area,
            'lia_volume_m3': v_init,
            'end_volume_m3': v_end,
            'lia_length_m': l_init,
            'end_length_m': l_end,
        })
    except Exception as e:
        print("{:<8} extract FAILED: {}".format(t0, e))

# 恢复temp_bias
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
    p = json.load(f)
p['temp_bias'] = original_tb
with open(os.path.join(gdir.dir, 'mb_calib.json'), 'w') as f:
    json.dump(p, f, indent=2)
print("\ntemp_bias已恢复")

# 保存结果
if results_rows:
    df = pd.DataFrame(results_rows)
    save_dir = os.path.join(cfg.PATHS['working_dir'], 'experiment_results_single')
    csv_path = os.path.join(save_dir, 'forward_results_{}.csv'.format(
        gdir.rgi_id.replace('.', '_')))
    df.to_csv(csv_path, index=False)
    print("\n结果保存到: {}".format(csv_path))

    # 找最佳t0
    best_idx = (df['sim_rgi_ratio'] - 1.0).abs().idxmin()
    best = df.loc[best_idx]
    print("\n最佳t0={}: sim/RGI={:.3f}, area_change={:+.1f}%, best_offset={:+.3f}".format(
        int(best['t0']), best['sim_rgi_ratio'], best['area_change_pct'], best['best_offset']))

In [ ]:
"""
可视化正演时间序列（修正版）
使用真实RGI观测面积而非LIA轮廓面积
"""
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pickle, os
from oggm import cfg

# ========== 加载数据 ==========
pkl_path = os.path.join(cfg.PATHS['working_dir'], 'experiment_results_single',
                        'window_offset_single_RGI60-15_00007.pkl')
with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

all_results = data['all_results']
windows = data['windows']
gdir = gdirs_lia[0]

# ===== 关键修正: 获取真实RGI观测面积 =====
uid = int(gdir.rgi_id.split('.')[-1])
main_rgi_id = lia_to_main_rgi[uid]  # LIA UniqueID → RGI ID

# 从gdirs_rgi中找到对应冰川的真实面积
rgi_gdir = [g for g in gdirs_rgi if g.rgi_id == main_rgi_id][0]
rgi_area = rgi_gdir.rgi_area_m2 / 1e6  # 真实RGI观测面积 (km2)
lia_outline_area = gdir.rgi_area_m2 / 1e6  # LIA轮廓面积 (km2)

print("冰川: {} → MainRGI: {}".format(gdir.rgi_id, main_rgi_id))
print("LIA轮廓面积: {:.2f} km², RGI观测面积: {:.2f} km²".format(lia_outline_area, rgi_area))

# ========== 读取各t0的正演时间序列 ==========
ts_data = {}
for t0 in windows:
    bo = all_results[t0]['best_offset']
    if np.isnan(bo):
        continue
    try:
        ds = xr.open_dataset(gdir.get_filepath('model_diagnostics',
                                                filesuffix='_forward_t0_{}'.format(t0)))
        n = len(ds['area_m2'].values)
        ts_data[t0] = {
            'years': np.arange(t0, t0 + n),
            'area_km2': ds['area_m2'].values / 1e6,
            'volume_km3': ds['volume_m3'].values / 1e9,
            'length_km': ds['length_m'].values / 1e3,
            'best_offset': bo,
        }
        ds.close()
    except Exception as e:
        print("t0={}: {}".format(t0, e))

# ========== 图1: 面积时间序列 ==========
fig, ax = plt.subplots(figsize=(12, 5))
cmap = plt.cm.viridis(np.linspace(0, 1, len(ts_data)))

for i, t0 in enumerate(sorted(ts_data.keys())):
    d = ts_data[t0]
    ax.plot(d['years'], d['area_km2'], color=cmap[i],
            label='t0={} (Δ={:+.2f}°C)'.format(t0, d['best_offset']), linewidth=1.5)

# RGI真实观测面积
ax.axhline(rgi_area, color='red', linestyle='--', linewidth=2,
           label='RGI observed ({:.2f} km²)'.format(rgi_area))
# LIA轮廓面积（参考）
ax.axhline(lia_outline_area, color='blue', linestyle=':', linewidth=1.5, alpha=0.5,
           label='LIA outline ({:.2f} km²)'.format(lia_outline_area))

ax.set_xlabel('Year')
ax.set_ylabel('Area (km²)')
ax.set_title('{} → {} — Area evolution'.format(gdir.rgi_id, main_rgi_id))
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ========== 图2: sim/RGI ratio vs t0 ==========
fig, ax = plt.subplots(figsize=(10, 4))

t0s = sorted(ts_data.keys())
ratios = [ts_data[t0]['area_km2'][-1] / rgi_area for t0 in t0s]

ax.plot(t0s, ratios, 'ko-', markersize=8)
ax.axhline(1.0, color='red', linestyle='--', label='ratio = 1.0')
ax.fill_between([min(t0s)-25, max(t0s)+25], 0.95, 1.05, alpha=0.15, color='green')

best_i = np.argmin(np.abs(np.array(ratios) - 1.0))
ax.annotate('Best: t0={}, ratio={:.3f}'.format(t0s[best_i], ratios[best_i]),
            xy=(t0s[best_i], ratios[best_i]), fontsize=10,
            xytext=(30, 15), textcoords='offset points',
            arrowprops=dict(arrowstyle='->'),
            bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

ax.set_xlabel('Starting window (t0)')
ax.set_ylabel('sim_2000 / RGI_observed area ratio')
ax.set_title('{} — sim/RGI ratio (using true RGI area)'.format(gdir.rgi_id))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ========== 汇总表 ==========
print("\n" + "=" * 85)
print("{:<6} {:>8} {:>10} {:>10} {:>12} {:>10}".format(
    "t0", "offset", "sim_LIA", "sim_2000", "RGI_obs", "sim/RGI"))
print("-" * 85)
for i, t0 in enumerate(t0s):
    d = ts_data[t0]
    sim_lia = d['area_km2'][0]
    sim_2000 = d['area_km2'][-1]
    r = sim_2000 / rgi_area
    mark = " ◀ BEST" if t0 == t0s[best_i] else ""
    print("{:<6} {:>+8.3f} {:>10.2f} {:>10.2f} {:>12.2f} {:>10.3f}{}".format(
        t0, d['best_offset'], sim_lia, sim_2000, rgi_area, r, mark))

In [ ]:
"""
batch_offset_search_parallel_test.py
=====================================
Jupyter Notebook 版 - 按 # %% 分cell复制运行
修改点：
  1. TEST_N = 20 只跑前20条（改为 None 全量跑）
  2. 多进程始终开启
  3. execute_entity_task 全部 try-except 防卡死
  4. task_timeout=600s
"""

# %% Cell 1 - 配置

import json, os, gc, time
import numpy as np
import xarray as xr
import pandas as pd
import pickle

from oggm import cfg, workflow, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

# ---- 测试开关 ----
TEST_N = 20  # 设为 None 则全量运行

# ---- 多进程 ----
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['task_timeout'] = 600


class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)


WINDOWS = [1350, 1500, 1650, 1800]
OFFSETS = [round(-i * 0.2, 1) for i in range(11)]
SPINUP_YEARS = 500
HALFSIZE = 30
CLIMATE_SUFFIX = '_ensemble_rec_ref'

SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'batch_offset_results')
os.makedirs(SAVE_DIR, exist_ok=True)

print("多进程={}, 进程数={}, timeout={}s".format(
    cfg.PARAMS['use_multiprocessing'], cfg.PARAMS['mp_processes'], cfg.PARAMS['task_timeout']))

# %% Cell 2 - 构建映射 & 筛选

print("[Step 1] 构建LIA→RGI面积映射...")

rgi_area_map = {}
rgi_id_map = {}
original_params = {}

for gdir in gdirs_lia:
    uid = int(gdir.rgi_id.split('.')[-1])
    try:
        main_rgi_id = lia_to_main_rgi[uid]
        rgi_gdir = [g for g in gdirs_rgi if g.rgi_id == main_rgi_id][0]
        rgi_area_map[gdir.rgi_id] = rgi_gdir.rgi_area_m2 / 1e6
        rgi_id_map[gdir.rgi_id] = main_rgi_id
    except:
        continue
    try:
        params = lia_mb_params_cru[uid]
        original_params[gdir.rgi_id] = {
            'temp_bias': params['temp_bias'],
            'melt_f': params['melt_f'],
            'prcp_fac': params['prcp_fac'],
        }
    except:
        pass

valid_gdirs = [g for g in gdirs_lia if g.rgi_id in rgi_area_map and g.rgi_id in original_params]
print("有效冰川: {} / {}".format(len(valid_gdirs), len(gdirs_lia)))

if TEST_N is not None:
    valid_gdirs = valid_gdirs[:TEST_N]
    print(">>> 测试模式: 取前 {} 条 <<<".format(len(valid_gdirs)))

print("运行: {} 冰川, {} 窗口 × {} offset = {} 轮".format(
    len(valid_gdirs), len(WINDOWS), len(OFFSETS), len(WINDOWS) * len(OFFSETS)))

# %% Cell 3 - 辅助函数

def batch_set_temp_bias(gdirs, offset):
    for gdir in gdirs:
        orig_tb = original_params[gdir.rgi_id]['temp_bias']
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = orig_tb + offset
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)


def batch_restore_temp_bias(gdirs):
    for gdir in gdirs:
        orig_tb = original_params[gdir.rgi_id]['temp_bias']
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = orig_tb
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)


def extract_results(gdirs, y0, offset):
    suffix = '_forward_w{}_o{}'.format(y0, str(offset).replace('.', 'p').replace('-', 'n'))
    results = []
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        try:
            fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
            if not os.path.exists(fp):
                continue
            ds = xr.open_dataset(fp)
            rgi_a = rgi_area_map.get(rgi_id, np.nan)
            end_area = float(ds['area_m2'].values[-1]) / 1e6
            lia_area = float(ds['area_m2'].values[0]) / 1e6
            results.append({
                'rgi_id': rgi_id,
                'main_rgi_id': rgi_id_map.get(rgi_id, ''),
                'y0': y0, 'offset': offset,
                'actual_temp_bias': original_params[rgi_id]['temp_bias'] + offset,
                'original_temp_bias': original_params[rgi_id]['temp_bias'],
                'melt_f': original_params[rgi_id]['melt_f'],
                'prcp_fac': original_params[rgi_id]['prcp_fac'],
                'lia_area_km2': lia_area,
                'end_area_km2': end_area,
                'rgi_area_km2': rgi_a,
                'sim_rgi_ratio': end_area / rgi_a if rgi_a > 0 else np.nan,
                'area_change_pct': (end_area - lia_area) / lia_area * 100,
                'lia_volume_km3': float(ds['volume_m3'].values[0]) / 1e9,
                'end_volume_km3': float(ds['volume_m3'].values[-1]) / 1e9,
                'lia_length_km': float(ds['length_m'].values[0]) / 1e3,
                'end_length_km': float(ds['length_m'].values[-1]) / 1e3,
            })
            ds.close()
        except Exception as e:
            print("    [WARN] 提取失败 {}: {}".format(rgi_id, e))
    return results

# %% Cell 4 - 批量运行（主循环）

print("[Step 2] 开始批量运行...")

all_results = []
failed_rounds = []
total_rounds = len(WINDOWS) * len(OFFSETS)
round_count = 0
t_start = time.time()

for y0 in WINDOWS:
    for offset in OFFSETS:
        round_count += 1
        t_round = time.time()

        suffix_spinup = '_spinup_w{}_o{}'.format(y0, str(offset).replace('.', 'p').replace('-', 'n'))
        suffix_forward = '_forward_w{}_o{}'.format(y0, str(offset).replace('.', 'p').replace('-', 'n'))

        print("\n--- [{}/{}] y0={}, offset={:+.1f} ---".format(
            round_count, total_rounds, y0, offset))

        batch_set_temp_bias(valid_gdirs, offset)

        # Spin-up
        print("  Spin-up...")
        try:
            execute_entity_task(
                tasks.run_constant_climate, valid_gdirs,
                nyears=SPINUP_YEARS, y0=y0, halfsize=HALFSIZE,
                climate_filename='gcm_data',
                climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                output_filesuffix=suffix_spinup,
                store_model_geometry=True,
            )
        except Exception as e:
            print("  [ERROR] Spin-up异常: {}".format(e))
            failed_rounds.append({'y0': y0, 'offset': offset, 'stage': 'spinup', 'error': str(e)})
            batch_restore_temp_bias(valid_gdirs)
            gc.collect()
            continue

        # 筛选成功
        spinup_ok = []
        for gdir in valid_gdirs:
            try:
                fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_spinup)
                if os.path.exists(fp):
                    spinup_ok.append(gdir)
            except:
                pass
        print("  Spin-up成功: {}/{}".format(len(spinup_ok), len(valid_gdirs)))

        if len(spinup_ok) == 0:
            batch_restore_temp_bias(valid_gdirs)
            gc.collect()
            continue

        # 正演
        print("  正演到2000...")
        try:
            execute_entity_task(
                tasks.run_from_climate_data, spinup_ok,
                min_ys=y0, ye=2000,
                climate_filename='gcm_data',
                climate_input_filesuffix=CLIMATE_SUFFIX,
                mb_model_class=MyMBModel,
                evolution_model=FluxBasedModel,
                init_model_filesuffix=suffix_spinup,
                output_filesuffix=suffix_forward,
            )
        except Exception as e:
            print("  [ERROR] 正演异常: {}".format(e))
            failed_rounds.append({'y0': y0, 'offset': offset, 'stage': 'forward', 'error': str(e)})

        # 提取
        round_results = extract_results(valid_gdirs, y0, offset)
        all_results.extend(round_results)

        # 恢复
        batch_restore_temp_bias(valid_gdirs)

        # 进度
        elapsed = time.time() - t_round
        remaining = (total_rounds - round_count) * ((time.time() - t_start) / round_count) / 3600
        print("  成功: {}, 耗时: {:.1f}min, 剩余: {:.1f}h".format(
            len(round_results), elapsed / 60, remaining))

        # checkpoint
        pd.DataFrame(all_results).to_csv(
            os.path.join(SAVE_DIR, 'results_checkpoint.csv'), index=False)
        gc.collect()

print("\n完成! 耗时: {:.1f}h, 结果: {}, 失败: {}".format(
    (time.time() - t_start) / 3600, len(all_results), len(failed_rounds)))

if failed_rounds:
    for fr in failed_rounds:
        print("  失败: y0={}, offset={:+.1f}, {}: {}".format(
            fr['y0'], fr['offset'], fr['stage'], fr['error'][:100]))

# %% Cell 5 - 汇总分析

print("[Step 3] 汇总分析...")

if len(all_results) == 0:
    print("无结果，跳过")
else:
    df = pd.DataFrame(all_results)
    df.to_csv(os.path.join(SAVE_DIR, 'all_results.csv'), index=False)

    print("\n{:<6} {:>8} {:>10} {:>10} {:>10} {:>10} {:>8}".format(
        "y0", "offset", "mean_rat", "med_rat", "std_rat", "n_good", "n_total"))
    print("-" * 72)

    summary_rows = []
    for y0 in WINDOWS:
        for offset in OFFSETS:
            sub = df[(df['y0'] == y0) & (df['offset'] == offset)]
            ratios = sub['sim_rgi_ratio'].dropna()
            if len(ratios) == 0:
                continue
            n_good = ((ratios >= 0.8) & (ratios <= 1.2)).sum()
            row = {
                'y0': y0, 'offset': offset,
                'mean_ratio': ratios.mean(), 'median_ratio': ratios.median(),
                'std_ratio': ratios.std(), 'n_good_80_120': int(n_good),
                'n_total': len(ratios),
                'pct_good': n_good / len(ratios) * 100,
            }
            summary_rows.append(row)
            print("{:<6} {:>+8.1f} {:>10.3f} {:>10.3f} {:>10.3f} {:>10} {:>8}".format(
                y0, offset, row['mean_ratio'], row['median_ratio'],
                row['std_ratio'], row['n_good_80_120'], row['n_total']))

    df_summary = pd.DataFrame(summary_rows)
    df_summary.to_csv(os.path.join(SAVE_DIR, 'summary_by_window_offset.csv'), index=False)

    # 每个窗口最优
    print("\n最优offset（每个窗口）:")
    for y0 in WINDOWS:
        sub = df_summary[df_summary['y0'] == y0]
        if len(sub) == 0:
            continue
        best_idx = (sub['median_ratio'] - 1.0).abs().idxmin()
        best = sub.loc[best_idx]
        print("  y0={}: offset={:+.1f}, median={:.3f}, pct_good={:.1f}%".format(
            y0, best['offset'], best['median_ratio'], best['pct_good']))

    # 全局最优
    print("\n全局最优offset:")
    global_best_offset = None
    global_best_score = 999
    for offset in OFFSETS:
        sub = df_summary[df_summary['offset'] == offset]
        if len(sub) == 0:
            continue
        avg_med = sub['median_ratio'].mean()
        avg_pct = sub['pct_good'].mean()
        score = abs(avg_med - 1.0)
        if score < global_best_score:
            global_best_score = score
            global_best_offset = offset
        print("  offset={:+.1f}: avg_median={:.3f}, avg_pct_good={:.1f}%".format(
            offset, avg_med, avg_pct))

    if global_best_offset is not None:
        print("\n>>> 全局最优: offset={:+.1f} <<<".format(global_best_offset))

    # pickle
    pkl_path = os.path.join(SAVE_DIR, 'batch_results.pkl')
    with open(pkl_path, 'wb') as f:
        pickle.dump({
            'all_results': all_results, 'summary': summary_rows,
            'failed_rounds': failed_rounds, 'windows': WINDOWS,
            'offsets': OFFSETS, 'rgi_area_map': rgi_area_map,
            'rgi_id_map': rgi_id_map, 'valid_glacier_count': len(valid_gdirs),
            'global_best_offset': global_best_offset, 'test_n': TEST_N,
        }, f)
    print("Pickle: {}".format(pkl_path))

print("\n实验完成!")

In [ ]:
"""
batch_offset_results 可视化（spinup + 正演拼接时间序列）
按 # %% 分cell在Jupyter中运行
"""

# %% Cell 1 - 加载数据

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os
from oggm import cfg

SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'batch_offset_results')

with open(os.path.join(SAVE_DIR, 'batch_results.pkl'), 'rb') as f:
    data = pickle.load(f)

df = pd.DataFrame(data['all_results'])
WINDOWS = data['windows']
OFFSETS = data['offsets']
rgi_area_map = data['rgi_area_map']
rgi_id_map = data['rgi_id_map']
global_best = data['global_best_offset']

# LIA轮廓面积
lia_area_map = {}
for gdir in valid_gdirs:
    lia_area_map[gdir.rgi_id] = gdir.rgi_area_m2 / 1e6

print("结果数: {}, 冰川数: {}".format(len(df), df['rgi_id'].nunique()))
print("全局最优offset: {:+.1f}".format(global_best))


# %% Cell 2 - 辅助函数（读spinup + 正演，拼接）

SPINUP_YEARS = 500

def load_spinup_timeseries(gdir, y0, offset):
    """读取spin-up时间序列"""
    suffix = '_spinup_w{}_o{}'.format(y0, str(offset).replace('.', 'p').replace('-', 'n'))
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
        if not os.path.exists(fp):
            return None
        ds = xr.open_dataset(fp)
        n = len(ds['area_m2'].values)
        # spinup的时间轴: y0 - SPINUP_YEARS 到 y0
        years = np.linspace(y0 - SPINUP_YEARS, y0, n)
        result = {
            'years': years,
            'area_km2': ds['area_m2'].values / 1e6,
            'volume_km3': ds['volume_m3'].values / 1e9,
            'length_km': ds['length_m'].values / 1e3,
        }
        ds.close()
        return result
    except:
        return None


def load_forward_timeseries(gdir, y0, offset):
    """读取正演时间序列"""
    suffix = '_forward_w{}_o{}'.format(y0, str(offset).replace('.', 'p').replace('-', 'n'))
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
        if not os.path.exists(fp):
            return None
        ds = xr.open_dataset(fp)
        n = len(ds['area_m2'].values)
        result = {
            'years': np.arange(y0, y0 + n),
            'area_km2': ds['area_m2'].values / 1e6,
            'volume_km3': ds['volume_m3'].values / 1e9,
            'length_km': ds['length_m'].values / 1e3,
        }
        ds.close()
        return result
    except:
        return None


def load_full_timeseries(gdir, y0, offset):
    """拼接 spinup + 正演，返回完整时间序列"""
    sp = load_spinup_timeseries(gdir, y0, offset)
    fw = load_forward_timeseries(gdir, y0, offset)

    if sp is None and fw is None:
        return None

    result = {}
    for var in ['area_km2', 'volume_km3', 'length_km']:
        parts_y = []
        parts_v = []
        if sp is not None:
            parts_y.append(sp['years'])
            parts_v.append(sp[var])
        if fw is not None:
            # 去掉正演第一个点（与spinup最后一个点重复）
            if sp is not None:
                parts_y.append(fw['years'][1:])
                parts_v.append(fw[var][1:])
            else:
                parts_y.append(fw['years'])
                parts_v.append(fw[var])

        result[var] = np.concatenate(parts_v)

    result['years'] = np.concatenate(parts_y) if sp is not None and fw is not None else \
                      np.concatenate([sp['years']]) if sp is not None else fw['years']

    # 修正：统一拼接years
    parts_y = []
    if sp is not None:
        parts_y.append(sp['years'])
    if fw is not None:
        if sp is not None:
            parts_y.append(fw['years'][1:])
        else:
            parts_y.append(fw['years'])
    result['years'] = np.concatenate(parts_y)

    # 记录spinup/forward分界点
    result['y0'] = y0

    return result


# %% Cell 3 - 图1: 20条冰川 × 多offset（固定y0=1500），spinup+正演拼接

PLOT_Y0 = 1500
PLOT_OFFSETS = [0, -0.2, -0.4, -0.6, -0.8, -1.0]

fig, axes = plt.subplots(4, 5, figsize=(28, 18))
axes = axes.flatten()

plot_idx = 0
for gdir in valid_gdirs:
    if plot_idx >= 20:
        break
    rgi_id = gdir.rgi_id
    rgi_area = rgi_area_map.get(rgi_id, np.nan)
    lia_area = lia_area_map.get(rgi_id, np.nan)
    if np.isnan(rgi_area):
        continue

    ax = axes[plot_idx]
    cmap = plt.cm.coolwarm(np.linspace(0, 1, len(PLOT_OFFSETS)))

    for j, offset in enumerate(PLOT_OFFSETS):
        ts = load_full_timeseries(gdir, PLOT_Y0, offset)
        if ts is None:
            continue
        lw = 2.0 if offset == global_best else 1.0
        alpha = 1.0 if offset == global_best else 0.6
        ax.plot(ts['years'], ts['area_km2'], color=cmap[j], linewidth=lw,
                alpha=alpha, label='Δ={:+.1f}'.format(offset))

    # spinup/forward分界线
    ax.axvline(PLOT_Y0, color='gray', linestyle='-', linewidth=0.8, alpha=0.5)

    # LIA + RGI参考线
    ax.axhline(lia_area, color='blue', linestyle=':', linewidth=1.8,
               label='LIA outline' if plot_idx == 0 else '')
    ax.axhline(rgi_area, color='red', linestyle='--', linewidth=1.8,
               label='RGI observed' if plot_idx == 0 else '')

    ax.set_title('{}\nLIA={:.1f}, RGI={:.1f} km²'.format(
        rgi_id.split('.')[-1], lia_area, rgi_area), fontsize=9)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

    if plot_idx == 0:
        ax.legend(fontsize=5, ncol=2, loc='upper right')

    plot_idx += 1

for i in range(plot_idx, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('Spinup + Forward (y0={}) | Blue:=LIA | Red--=RGI | Gray|=spinup/forward boundary'.format(PLOT_Y0),
             fontsize=13, fontweight='bold')
fig.supxlabel('Year', fontsize=12)
fig.supylabel('Area (km²)', fontsize=12)
plt.tight_layout()
plt.show()


# %% Cell 4 - 图2: 20条冰川 × 多窗口（固定offset=最优），spinup+正演拼接

PLOT_OFFSET = global_best

fig, axes = plt.subplots(4, 5, figsize=(28, 18))
axes = axes.flatten()

plot_idx = 0
for gdir in valid_gdirs:
    if plot_idx >= 20:
        break
    rgi_id = gdir.rgi_id
    rgi_area = rgi_area_map.get(rgi_id, np.nan)
    lia_area = lia_area_map.get(rgi_id, np.nan)
    if np.isnan(rgi_area):
        continue

    ax = axes[plot_idx]
    cmap = plt.cm.viridis(np.linspace(0, 1, len(WINDOWS)))

    for j, y0 in enumerate(WINDOWS):
        ts = load_full_timeseries(gdir, y0, PLOT_OFFSET)
        if ts is None:
            continue
        ax.plot(ts['years'], ts['area_km2'], color=cmap[j], linewidth=1.5,
                label='y0={}'.format(y0))
        # 每个窗口的分界线
        ax.axvline(y0, color=cmap[j], linestyle=':', linewidth=0.6, alpha=0.4)

    ax.axhline(lia_area, color='blue', linestyle=':', linewidth=1.8,
               label='LIA outline' if plot_idx == 0 else '')
    ax.axhline(rgi_area, color='red', linestyle='--', linewidth=1.8,
               label='RGI observed' if plot_idx == 0 else '')

    ax.set_title('{}\nLIA={:.1f}, RGI={:.1f} km²'.format(
        rgi_id.split('.')[-1], lia_area, rgi_area), fontsize=9)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

    if plot_idx == 0:
        ax.legend(fontsize=6, loc='upper right')

    plot_idx += 1

for i in range(plot_idx, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('Spinup + Forward (offset={:+.1f}) | Blue:=LIA | Red--=RGI'.format(PLOT_OFFSET),
             fontsize=13, fontweight='bold')
fig.supxlabel('Year', fontsize=12)
fig.supylabel('Area (km²)', fontsize=12)
plt.tight_layout()
plt.show()


# %% Cell 5 - 图3: 热力图

summary = pd.DataFrame(data['summary'])

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

pivot = summary.pivot(index='y0', columns='offset', values='median_ratio')
im1 = axes[0].imshow(pivot.values, cmap='RdYlGn_r', aspect='auto', vmin=0.8, vmax=2.0)
axes[0].set_xticks(range(len(pivot.columns)))
axes[0].set_xticklabels(['{:+.1f}'.format(x) for x in pivot.columns], fontsize=8)
axes[0].set_yticks(range(len(pivot.index)))
axes[0].set_yticklabels(pivot.index)
axes[0].set_xlabel('Offset (K)')
axes[0].set_ylabel('Window (y0)')
axes[0].set_title('Median sim/RGI ratio')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        axes[0].text(j, i, '{:.2f}'.format(v), ha='center', va='center', fontsize=8,
                     color='white' if v > 1.5 else 'black')
plt.colorbar(im1, ax=axes[0])

pivot_pct = summary.pivot(index='y0', columns='offset', values='pct_good')
im2 = axes[1].imshow(pivot_pct.values, cmap='YlGn', aspect='auto', vmin=0, vmax=100)
axes[1].set_xticks(range(len(pivot_pct.columns)))
axes[1].set_xticklabels(['{:+.1f}'.format(x) for x in pivot_pct.columns], fontsize=8)
axes[1].set_yticks(range(len(pivot_pct.index)))
axes[1].set_yticklabels(pivot_pct.index)
axes[1].set_xlabel('Offset (K)')
axes[1].set_ylabel('Window (y0)')
axes[1].set_title('% glaciers with ratio in [0.8, 1.2]')
for i in range(len(pivot_pct.index)):
    for j in range(len(pivot_pct.columns)):
        v = pivot_pct.values[i, j]
        axes[1].text(j, i, '{:.0f}%'.format(v), ha='center', va='center', fontsize=8)
plt.colorbar(im2, ax=axes[1])

plt.suptitle('Batch offset search summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


# %% Cell 6 - 图4: ratio柱状图 + LIA/RGI/Sim面积三组对比

sub = df[(df['offset'] == global_best) & (df['y0'] == 1500)].copy()
sub = sub.sort_values('sim_rgi_ratio').reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

x = range(len(sub))
colors = ['green' if 0.8 <= r <= 1.2 else 'orange' if 0.6 <= r <= 1.4 else 'red'
          for r in sub['sim_rgi_ratio']]
ax1.bar(x, sub['sim_rgi_ratio'], color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax1.axhline(1.0, color='red', linestyle='--', linewidth=1.5)
ax1.fill_between([-0.5, len(sub)-0.5], 0.8, 1.2, alpha=0.1, color='green')
ax1.set_xticks(x)
ax1.set_xticklabels([rid.split('.')[-1] for rid in sub['rgi_id']], rotation=45, fontsize=8)
ax1.set_ylabel('sim_2000 / RGI area')
ax1.set_title('Per-glacier ratio (y0=1500, offset={:+.1f})'.format(global_best))
ax1.grid(True, alpha=0.3, axis='y')

bar_width = 0.25
x_pos = np.arange(len(sub))
lia_areas = [lia_area_map.get(rid, 0) for rid in sub['rgi_id']]

ax2.bar(x_pos - bar_width, lia_areas, bar_width, label='LIA outline', color='blue', alpha=0.7)
ax2.bar(x_pos, sub['rgi_area_km2'].values, bar_width, label='RGI observed', color='red', alpha=0.7)
ax2.bar(x_pos + bar_width, sub['end_area_km2'].values, bar_width, label='Sim 2000', color='green', alpha=0.7)

ax2.set_xticks(x_pos)
ax2.set_xticklabels([rid.split('.')[-1] for rid in sub['rgi_id']], rotation=45, fontsize=8)
ax2.set_ylabel('Area (km²)')
ax2.set_title('LIA outline vs RGI observed vs Simulated 2000')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


# %% Cell 7 - 图5: 代表性冰川三合一（spinup+正演），面积+体积+长度

sub_best = df[(df['offset'] == global_best) & (df['y0'] == 1500)]
best_glacier_id = sub_best.loc[(sub_best['sim_rgi_ratio'] - 1.0).abs().idxmin(), 'rgi_id']
best_gdir = [g for g in valid_gdirs if g.rgi_id == best_glacier_id][0]
rgi_area = rgi_area_map[best_glacier_id]
lia_area = lia_area_map[best_glacier_id]

print("展示冰川: {} (ratio最接近1.0)".format(best_glacier_id))
print("LIA面积: {:.2f} km², RGI面积: {:.2f} km²".format(lia_area, rgi_area))

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
cmap = plt.cm.coolwarm(np.linspace(0, 1, len(OFFSETS)))

for j, offset in enumerate(OFFSETS):
    ts = load_full_timeseries(best_gdir, 1500, offset)
    if ts is None:
        continue
    lbl = 'Δ={:+.1f}'.format(offset)
    lw = 2.5 if offset == global_best else 0.8
    alpha = 1.0 if offset == global_best else 0.5

    axes[0].plot(ts['years'], ts['area_km2'], color=cmap[j], linewidth=lw, alpha=alpha, label=lbl)
    axes[1].plot(ts['years'], ts['volume_km3'], color=cmap[j], linewidth=lw, alpha=alpha, label=lbl)
    axes[2].plot(ts['years'], ts['length_km'], color=cmap[j], linewidth=lw, alpha=alpha, label=lbl)

# 分界线
axes[0].axvline(1500, color='gray', linestyle='-', linewidth=1, alpha=0.5, label='spinup→forward')
axes[1].axvline(1500, color='gray', linestyle='-', linewidth=1, alpha=0.5)
axes[2].axvline(1500, color='gray', linestyle='-', linewidth=1, alpha=0.5)

# 面积参考线
axes[0].axhline(lia_area, color='blue', linestyle=':', linewidth=2, label='LIA outline')
axes[0].axhline(rgi_area, color='red', linestyle='--', linewidth=2, label='RGI observed')

axes[0].set_ylabel('Area (km²)')
axes[0].set_title('{} → {} | Spinup(500yr) + Forward → 2000 | LIA={:.1f}, RGI={:.1f} km²'.format(
    best_glacier_id, rgi_id_map.get(best_glacier_id, ''), lia_area, rgi_area))
axes[0].legend(fontsize=7, ncol=4)
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel('Volume (km³)')
axes[1].grid(True, alpha=0.3)

axes[2].set_ylabel('Length (km)')
axes[2].set_xlabel('Year')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# %% Cell 8 - 汇总表

print("\n" + "=" * 100)
print("{:<12} {:>8} {:>10} {:>10} {:>10} {:>10} {:>10}".format(
    "glacier", "offset", "LIA_area", "sim_2000", "RGI_obs", "sim/RGI", "LIA/RGI"))
print("-" * 100)

sub = df[(df['offset'] == global_best) & (df['y0'] == 1500)].sort_values('sim_rgi_ratio')
for _, row in sub.iterrows():
    lia_a = lia_area_map.get(row['rgi_id'], np.nan)
    lia_rgi = lia_a / row['rgi_area_km2'] if row['rgi_area_km2'] > 0 else np.nan
    mark = " *" if 0.8 <= row['sim_rgi_ratio'] <= 1.2 else ""
    print("{:<12} {:>+8.1f} {:>10.2f} {:>10.2f} {:>10.2f} {:>10.3f} {:>10.3f}{}".format(
        row['rgi_id'].split('.')[-1], global_best, lia_a,
        row['end_area_km2'], row['rgi_area_km2'],
        row['sim_rgi_ratio'], lia_rgi, mark))

print("\n* = sim/RGI在[0.8, 1.2]范围内")

In [ ]:
"""
melt_f_scale_spinup_test.py
============================
验证 melt_f 缩放方案：
  1. 每条冰川用各自的平衡 scale 缩放 melt_f（使 LIA 几何上 MB≈0）
  2. spin-up 300年（恒定气候）
  3. 正演到 2000年
  4. 对比终点面积 vs RGI 观测面积
"""

# %% Cell 1 - 配置

import json, os, gc, time
import numpy as np
import xarray as xr
import pandas as pd

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ---- 参数 ----
TEST_N = 20
Y0 = 1500              # spin-up 用的气候中心年
HALFSIZE = 30
SPINUP_YEARS = 300
CLIMATE_SUFFIX = '_ensemble_rec_ref'
SCALE_FACTORS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

cfg.PARAMS['use_multiprocessing'] = False   # 20条用单进程更稳定
cfg.PARAMS['continue_on_error'] = True

SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'meltf_scale_results')
os.makedirs(SAVE_DIR, exist_ok=True)

# %% Cell 2 - 筛选冰川 & 计算每条冰川的平衡 scale

print("[Step 1] 筛选冰川 & 计算平衡 scale...")

# 筛选有 model_flowlines 的冰川
test_gdirs = [g for g in gdirs_lia
              if g.rgi_id in original_params
              and os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl'))][:TEST_N]

print("测试冰川: {} 条".format(len(test_gdirs)))

# 计算每条冰川的平衡 scale（复用之前的诊断逻辑）
eq_scales = {}

for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    params = original_params[rgi_id]
    fls = gdir.read_pickle('model_flowlines')
    original_melt_f = params['melt_f']

    mbs = []
    for scale in SCALE_FACTORS:
        try:
            mb = MyMBModel(
                gdir, filename='gcm_data',
                input_filesuffix=CLIMATE_SUFFIX,
                melt_f=original_melt_f * scale,
                temp_bias=params['temp_bias'],
                prcp_fac=params['prcp_fac'],
            )
            avg = np.mean([mb.get_specific_mb(fls=fls, year=yr)
                           for yr in range(Y0 - HALFSIZE, Y0 + HALFSIZE + 1)])
        except:
            avg = np.nan
        mbs.append(avg)

    # 线性插值找平衡点
    found = False
    for i in range(len(mbs) - 1):
        if not np.isnan(mbs[i]) and not np.isnan(mbs[i+1]):
            if mbs[i] >= 0 >= mbs[i+1]:
                s0, s1 = SCALE_FACTORS[i], SCALE_FACTORS[i+1]
                frac = -mbs[i] / (mbs[i+1] - mbs[i])
                eq_scales[rgi_id] = s0 + frac * (s1 - s0)
                found = True
                break
    if not found:
        eq_scales[rgi_id] = 0.6  # fallback

    uid = rgi_id.split('.')[-1]
    print("  {} : scale={:.3f}, melt_f={:.2f} → {:.2f}".format(
        uid, eq_scales[rgi_id],
        original_melt_f, original_melt_f * eq_scales[rgi_id]))

# %% Cell 3 - 写入缩放后的 melt_f & 运行

print("\n[Step 2] 写入缩放后的 melt_f...")

def batch_set_meltf_scale(gdirs, scales_dict):
    """将每条冰川的 melt_f 缩放写入 mb_calib.json"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        scale = scales_dict[rgi_id]
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['melt_f'] = original_params[rgi_id]['melt_f'] * scale
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

def batch_restore_meltf(gdirs):
    """恢复原始 melt_f"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['melt_f'] = original_params[rgi_id]['melt_f']
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

# 写入
batch_set_meltf_scale(test_gdirs, eq_scales)

suffix_spinup = '_spinup_meltf_scale'
suffix_forward = '_forward_meltf_scale'

# Spin-up
print("\n[Step 3] Spin-up ({}年, y0={})...".format(SPINUP_YEARS, Y0))
t0 = time.time()
try:
    execute_entity_task(
        tasks.run_constant_climate, test_gdirs,
        nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=suffix_spinup,
        store_model_geometry=True,
    )
except Exception as e:
    print("  Spin-up异常: {}".format(e))
print("  Spin-up耗时: {:.1f}min".format((time.time() - t0) / 60))

# 筛选成功
spinup_ok = []
for gdir in test_gdirs:
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_spinup)
        if os.path.exists(fp):
            spinup_ok.append(gdir)
    except:
        pass
print("  Spin-up成功: {}/{}".format(len(spinup_ok), len(test_gdirs)))

# 正演
print("\n[Step 4] 正演 {} → 2000...".format(Y0))
t0 = time.time()
try:
    execute_entity_task(
        tasks.run_from_climate_data, spinup_ok,
        min_ys=Y0, ye=2000,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        init_model_filesuffix=suffix_spinup,
        output_filesuffix=suffix_forward,
    )
except Exception as e:
    print("  正演异常: {}".format(e))
print("  正演耗时: {:.1f}min".format((time.time() - t0) / 60))

# 恢复原始参数
batch_restore_meltf(test_gdirs)

# %% Cell 4 - 提取结果 & 分析

print("\n[Step 5] 提取结果...")

results = []
for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    uid = rgi_id.split('.')[-1]
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_forward)
        if not os.path.exists(fp):
            continue
        ds = xr.open_dataset(fp)

        rgi_a = rgi_area_map.get(rgi_id, np.nan)
        end_area = float(ds['area_m2'].values[-1]) / 1e6
        lia_area = float(ds['area_m2'].values[0]) / 1e6
        spinup_area = lia_area  # 正演起点 = spin-up 终点

        # 也读 spin-up 起点面积（LIA 原始面积）
        fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_spinup)
        ds_sp = xr.open_dataset(fp_sp)
        lia_init_area = float(ds_sp['area_m2'].values[0]) / 1e6
        spinup_end_area = float(ds_sp['area_m2'].values[-1]) / 1e6
        ds_sp.close()

        ratio = end_area / rgi_a if rgi_a > 0 else np.nan
        results.append({
            'uid': uid,
            'rgi_id': rgi_id,
            'scale': eq_scales[rgi_id],
            'orig_melt_f': original_params[rgi_id]['melt_f'],
            'scaled_melt_f': original_params[rgi_id]['melt_f'] * eq_scales[rgi_id],
            'lia_init_area': lia_init_area,
            'spinup_end_area': spinup_end_area,
            'spinup_change_pct': (spinup_end_area - lia_init_area) / lia_init_area * 100,
            'forward_end_area': end_area,
            'rgi_area': rgi_a,
            'sim_rgi_ratio': ratio,
        })
        ds.close()
    except Exception as e:
        print("  提取失败 {}: {}".format(uid, e))

df = pd.DataFrame(results)

# 打印结果
print("\n" + "=" * 110)
print("{:<8} {:>6} {:>8} {:>8} {:>10} {:>10} {:>8} {:>10} {:>8} {:>8}".format(
    "UID", "scale", "orig_mf", "new_mf",
    "LIA_init", "SP_end", "SP_chg%",
    "FW_end", "RGI", "ratio"))
print("-" * 110)

for _, r in df.iterrows():
    print("{:<8} {:>6.2f} {:>8.2f} {:>8.2f} {:>10.2f} {:>10.2f} {:>8.1f} {:>10.2f} {:>8.2f} {:>8.3f}".format(
        r['uid'], r['scale'], r['orig_melt_f'], r['scaled_melt_f'],
        r['lia_init_area'], r['spinup_end_area'], r['spinup_change_pct'],
        r['forward_end_area'], r['rgi_area'], r['sim_rgi_ratio']))

print("-" * 110)
ratios = df['sim_rgi_ratio'].dropna()
n_good = ((ratios >= 0.8) & (ratios <= 1.2)).sum()
print("均值 ratio: {:.3f}  中位数: {:.3f}  std: {:.3f}".format(
    ratios.mean(), ratios.median(), ratios.std()))
print("ratio 在 [0.8, 1.2] 内: {}/{} = {:.1f}%".format(
    n_good, len(ratios), n_good / len(ratios) * 100))
print("spin-up 面积变化均值: {:.1f}%".format(df['spinup_change_pct'].mean()))

df.to_csv(os.path.join(SAVE_DIR, 'meltf_scale_test_results.csv'), index=False)
print("\n结果已保存: {}".format(SAVE_DIR))

In [ ]:
"""
temp_offset_two_strategies_v3.py
=================================
逐冰川 temp_offset，对比两种正演策略：
  策略A: spin-up + 正演 都保留 offset（类似 melt_f 方案）
  策略B: spin-up 加 offset，正演去掉 offset（让真实气候驱动退缩）

流程：
  Step 0: 重建派生变量
  Step 1: 逐冰川扫描offset → 线性插值找 eq_offset（使MB≈0）
  Step 2: 用 eq_offset 做共用 spin-up（300年）
  Step 3A: 策略A 正演（保留offset）
  Step 3B: 策略B 正演（去掉offset）
  Step 4: 对比分析 & 保存

改进：
  - 多进程 + continue_on_error 防卡死
  - 排除极端offset冰川
  - 借鉴单冰川脚本的提取和保存结构
  - 每步打印耗时和成功率
"""

# %% Cell 1 - 配置与导入

import json, os, gc, time
import numpy as np
import xarray as xr
import pandas as pd
import pickle

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ---------- 实验参数 ----------
TEST_N = 20                     # 测试冰川数量
Y0 = 1500                       # spin-up 中心年
HALFSIZE = 30                    # 气候窗口半宽
SPINUP_YEARS = 300               # spin-up 年数
CLIMATE_SUFFIX = '_ensemble_rec_ref'

# offset 扫描档位（用于找 eq_offset）
OFFSETS_SCAN = [0, -0.5, -1.0, -1.5, -2.0, -2.5, -3.0]

# 排除极端 offset 的阈值
MAX_OFFSET_ABS = 2.5  # |offset| > 2.5K 的冰川排除

# ---------- 多进程配置 ----------
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15
cfg.PARAMS['continue_on_error'] = True


SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'offset_two_strategies')
os.makedirs(SAVE_DIR, exist_ok=True)

# %% Cell 2 - 重建派生变量
# 假设 gdirs_lia, gdirs_rgi, lia_to_main_rgi, lia_mb_params_cru 已在环境中

print("=" * 70)
print("逐冰川 temp_offset 两策略对比实验")
print("=" * 70)

rgi_area_map = {}
rgi_id_map = {}
original_params = {}

for gdir in gdirs_lia:
    uid = int(gdir.rgi_id.split('.')[-1])
    try:
        main_rgi_id = lia_to_main_rgi[uid]
        rgi_gdir = [g for g in gdirs_rgi if g.rgi_id == main_rgi_id][0]
        rgi_area_map[gdir.rgi_id] = rgi_gdir.rgi_area_m2 / 1e6
        rgi_id_map[gdir.rgi_id] = main_rgi_id
    except:
        continue
    try:
        params = lia_mb_params_cru[uid]
        original_params[gdir.rgi_id] = {
            'temp_bias': params['temp_bias'],
            'melt_f':    params['melt_f'],
            'prcp_fac':  params['prcp_fac'],
        }
    except:
        pass

print("rgi_area_map : {}".format(len(rgi_area_map)))
print("original_params: {}".format(len(original_params)))

# %% Cell 3 - 筛选冰川 & 计算逐冰川平衡 offset

print("\n[Step 1] 筛选冰川 & 计算平衡 offset...")

test_gdirs = [g for g in gdirs_lia
              if g.rgi_id in original_params
              and g.rgi_id in rgi_area_map
              and os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl'))][:TEST_N]

print("测试冰川: {} 条".format(len(test_gdirs)))
print("扫描 offset: {}".format(OFFSETS_SCAN))

t_step1 = time.time()
eq_offsets = {}

for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    uid = rgi_id.split('.')[-1]
    params = original_params[rgi_id]
    fls = gdir.read_pickle('model_flowlines')

    mbs = []
    for off in OFFSETS_SCAN:
        try:
            mb = MyMBModel(
                gdir, filename='gcm_data',
                input_filesuffix=CLIMATE_SUFFIX,
                melt_f=params['melt_f'],
                temp_bias=params['temp_bias'] + off,
                prcp_fac=params['prcp_fac'],
            )
            avg = np.mean([mb.get_specific_mb(fls=fls, year=yr)
                           for yr in range(Y0 - HALFSIZE, Y0 + HALFSIZE + 1)])
        except:
            avg = np.nan
        mbs.append(avg)

    # 线性插值找平衡点（MB从正变负或负变正的交叉点）
    found = False
    for i in range(len(mbs) - 1):
        if not np.isnan(mbs[i]) and not np.isnan(mbs[i+1]):
            if mbs[i] <= 0 <= mbs[i+1] or mbs[i] >= 0 >= mbs[i+1]:
                o0, o1 = OFFSETS_SCAN[i], OFFSETS_SCAN[i+1]
                frac = -mbs[i] / (mbs[i+1] - mbs[i])
                eq_offsets[rgi_id] = o0 + frac * (o1 - o0)
                found = True
                break
    if not found:
        eq_offsets[rgi_id] = -1.5  # fallback

    print("  {} : offset={:+.2f}K  MB={}".format(
        uid, eq_offsets[rgi_id],
        '  '.join(['{:+.0f}'.format(m) if not np.isnan(m) else 'nan' for m in mbs])))

print("  Step1 耗时: {:.1f}min".format((time.time() - t_step1) / 60))

# %% Cell 4 - 排除极端 offset 冰川

print("\n[Step 1.5] 排除极端 offset 冰川 (|offset| > {:.1f}K)...".format(MAX_OFFSET_ABS))

test_gdirs_safe = [g for g in test_gdirs
                   if abs(eq_offsets.get(g.rgi_id, -999)) <= MAX_OFFSET_ABS]

excluded = [g.rgi_id.split('.')[-1] for g in test_gdirs
            if g not in test_gdirs_safe]
if excluded:
    print("  排除 {} 条: {}".format(len(excluded), ', '.join(excluded)))
print("  保留 {} 条冰川进入 spin-up".format(len(test_gdirs_safe)))

# %% Cell 5 - 辅助函数

def set_per_glacier_offset(gdirs, offsets_dict):
    """每条冰川写入各自的 temp_bias + offset 到 mb_calib.json"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        off = offsets_dict[rgi_id]
        orig_tb = original_params[rgi_id]['temp_bias']
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = orig_tb + off
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

def restore_original_params(gdirs):
    """恢复原始参数到 mb_calib.json"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = original_params[rgi_id]['temp_bias']
        p['melt_f'] = original_params[rgi_id]['melt_f']
        p['prcp_fac'] = original_params[rgi_id]['prcp_fac']
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

def extract_forward_results(gdirs, suffix_spinup, suffix_forward, strategy_name):
    """提取正演结果，借鉴单冰川脚本的详细字段"""
    results = []
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        uid = rgi_id.split('.')[-1]
        try:
            fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=suffix_forward)
            if not os.path.exists(fp_fw):
                continue
            ds_fw = xr.open_dataset(fp_fw)
            end_area = float(ds_fw['area_m2'].values[-1]) / 1e6
            end_vol = float(ds_fw['volume_m3'].values[-1]) / 1e9
            end_len = float(ds_fw['length_m'].values[-1]) / 1e3
            ds_fw.close()

            fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_spinup)
            ds_sp = xr.open_dataset(fp_sp)
            lia_init_area = float(ds_sp['area_m2'].values[0]) / 1e6
            spinup_end_area = float(ds_sp['area_m2'].values[-1]) / 1e6
            spinup_end_vol = float(ds_sp['volume_m3'].values[-1]) / 1e9
            ds_sp.close()

            rgi_a = rgi_area_map.get(rgi_id, np.nan)
            results.append({
                'uid': uid,
                'rgi_id': rgi_id,
                'strategy': strategy_name,
                'offset': eq_offsets[rgi_id],
                'lia_init_area': lia_init_area,
                'spinup_end_area': spinup_end_area,
                'spinup_chg_pct': (spinup_end_area - lia_init_area) / lia_init_area * 100,
                'spinup_end_vol_km3': spinup_end_vol,
                'forward_end_area': end_area,
                'forward_end_vol_km3': end_vol,
                'forward_end_len_km': end_len,
                'rgi_area': rgi_a,
                'sim_rgi_ratio': end_area / rgi_a if rgi_a > 0 else np.nan,
                'melt_f': original_params[rgi_id]['melt_f'],
                'temp_bias_original': original_params[rgi_id]['temp_bias'],
                'prcp_fac': original_params[rgi_id]['prcp_fac'],
            })
        except Exception as e:
            print("    提取失败 {}: {}".format(uid, e))
    return results

# %% Cell 6 - 共用 Spin-up

SUFFIX_SPINUP = '_spinup_perglacier_offset'

print("\n[Step 2] Spin-up (共用, {}年, y0={}, {}条冰川)...".format(
    SPINUP_YEARS, Y0, len(test_gdirs_safe)))

set_per_glacier_offset(test_gdirs_safe, eq_offsets)

t0 = time.time()
try:
    execute_entity_task(
        tasks.run_constant_climate, test_gdirs_safe,
        nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=SUFFIX_SPINUP,
        store_model_geometry=True,  # 不存几何，省内存
    )
except Exception as e:
    print("  Spin-up异常: {}".format(e))
print("  Spin-up耗时: {:.1f}min".format((time.time() - t0) / 60))

# 检查哪些成功了
spinup_ok = []
for gdir in test_gdirs_safe:
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_SPINUP)
        if os.path.exists(fp):
            spinup_ok.append(gdir)
    except:
        pass
print("  Spin-up成功: {}/{}".format(len(spinup_ok), len(test_gdirs_safe)))

# 打印 spin-up 面积变化
print("\n  {:<8} {:>8} {:>10} {:>10} {:>8}".format(
    "UID", "offset", "init_km2", "end_km2", "chg%"))
print("  " + "-" * 50)
for gdir in spinup_ok:
    uid = gdir.rgi_id.split('.')[-1]
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_SPINUP)
        ds = xr.open_dataset(fp)
        a0 = float(ds['area_m2'].values[0]) / 1e6
        ae = float(ds['area_m2'].values[-1]) / 1e6
        pct = (ae - a0) / a0 * 100
        ds.close()
        print("  {:<8} {:>+8.2f} {:>10.2f} {:>10.2f} {:>+8.1f}%".format(
            uid, eq_offsets[gdir.rgi_id], a0, ae, pct))
    except:
        pass

# %% Cell 7 - 策略A: 正演保留 offset

print("\n[Step 3A] 策略A: 正演保留 offset ({} 条冰川)...".format(len(spinup_ok)))
SUFFIX_FW_A = '_forward_keep_offset'

# offset 已在 mb_calib.json 中，直接正演
t0 = time.time()
try:
    execute_entity_task(
        tasks.run_from_climate_data, spinup_ok,
        min_ys=Y0, ye=2000,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        init_model_filesuffix=SUFFIX_SPINUP,
        output_filesuffix=SUFFIX_FW_A,
    )
except Exception as e:
    print("  正演A异常: {}".format(e))
print("  策略A耗时: {:.1f}min".format((time.time() - t0) / 60))

results_a = extract_forward_results(spinup_ok, SUFFIX_SPINUP, SUFFIX_FW_A, 'A_keep_offset')
print("  策略A成功提取: {} 条".format(len(results_a)))

# %% Cell 8 - 策略B: 正演去掉 offset

print("\n[Step 3B] 策略B: 正演去掉 offset ({} 条冰川)...".format(len(spinup_ok)))
SUFFIX_FW_B = '_forward_remove_offset'

# 恢复原始参数（去掉 offset）
restore_original_params(spinup_ok)

t0 = time.time()
try:
    execute_entity_task(
        tasks.run_from_climate_data, spinup_ok,
        min_ys=Y0, ye=2000,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        init_model_filesuffix=SUFFIX_SPINUP,
        output_filesuffix=SUFFIX_FW_B,
    )
except Exception as e:
    print("  正演B异常: {}".format(e))
print("  策略B耗时: {:.1f}min".format((time.time() - t0) / 60))

results_b = extract_forward_results(spinup_ok, SUFFIX_SPINUP, SUFFIX_FW_B, 'B_remove_offset')
print("  策略B成功提取: {} 条".format(len(results_b)))

# 恢复参数（安全）
restore_original_params(test_gdirs_safe)

# %% Cell 9 - 对比分析

print("\n" + "=" * 100)
print("[Step 4] 对比分析")
print("=" * 100)

df_a = pd.DataFrame(results_a)
df_b = pd.DataFrame(results_b)

# 逐冰川对比表
print("\n{:<8} {:>8} {:>10} {:>12} {:>12} {:>8} {:>10} {:>10}".format(
    "UID", "offset", "LIA", "SP_end", "A_end", "B_end", "RGI",
    "A_ratio", "B_ratio"))
print("-" * 100)

all_uids = sorted(set(
    (df_a['uid'].tolist() if len(df_a) > 0 else []) +
    (df_b['uid'].tolist() if len(df_b) > 0 else [])))

for uid in all_uids:
    ra = df_a[df_a['uid'] == uid] if len(df_a) > 0 else pd.DataFrame()
    rb = df_b[df_b['uid'] == uid] if len(df_b) > 0 else pd.DataFrame()

    if len(ra) == 0 and len(rb) == 0:
        continue

    ref = ra.iloc[0] if len(ra) > 0 else rb.iloc[0]
    a_end = ra.iloc[0]['forward_end_area'] if len(ra) > 0 else np.nan
    b_end = rb.iloc[0]['forward_end_area'] if len(rb) > 0 else np.nan
    a_rat = ra.iloc[0]['sim_rgi_ratio'] if len(ra) > 0 else np.nan
    b_rat = rb.iloc[0]['sim_rgi_ratio'] if len(rb) > 0 else np.nan

    print("{:<8} {:>+8.2f} {:>8.2f} {:>10.2f} {:>12.2f} {:>12.2f} {:>8.2f} {:>10.3f} {:>10.3f}".format(
        uid, ref['offset'], ref['lia_init_area'], ref['spinup_end_area'],
        a_end if not np.isnan(a_end) else 0,
        b_end if not np.isnan(b_end) else 0,
        ref['rgi_area'],
        a_rat if not np.isnan(a_rat) else 0,
        b_rat if not np.isnan(b_rat) else 0))

print("-" * 100)

# 汇总统计
for label, df_x in [('A (保留offset)', df_a), ('B (去掉offset)', df_b)]:
    if len(df_x) == 0:
        print("\n策略 {}: 无结果".format(label))
        continue
    ratios = df_x['sim_rgi_ratio'].dropna()
    n_good = ((ratios >= 0.8) & (ratios <= 1.2)).sum()
    print("\n策略 {}:".format(label))
    print("  均值ratio={:.3f}  中位数={:.3f}  std={:.3f}".format(
        ratios.mean(), ratios.median(), ratios.std()))
    print("  [0.8-1.2]内: {}/{}={:.1f}%".format(
        n_good, len(ratios), n_good / len(ratios) * 100 if len(ratios) > 0 else 0))
    print("  spin-up面积变化均值: {:.1f}%".format(df_x['spinup_chg_pct'].mean()))

# %% Cell 10 - 保存结果

# CSV
df_all = pd.concat([df_a, df_b])
csv_path = os.path.join(SAVE_DIR, 'two_strategies_results.csv')
df_all.to_csv(csv_path, index=False)
print("\nCSV 已保存: {}".format(csv_path))

# Pickle（包含完整中间信息）
pkl_data = {
    'eq_offsets': eq_offsets,
    'results_a': results_a,
    'results_b': results_b,
    'test_gdirs_ids': [g.rgi_id for g in test_gdirs],
    'safe_gdirs_ids': [g.rgi_id for g in test_gdirs_safe],
    'spinup_ok_ids': [g.rgi_id for g in spinup_ok],
    'config': {
        'TEST_N': TEST_N,
        'Y0': Y0,
        'HALFSIZE': HALFSIZE,
        'SPINUP_YEARS': SPINUP_YEARS,
        'OFFSETS_SCAN': OFFSETS_SCAN,
        'MAX_OFFSET_ABS': MAX_OFFSET_ABS,
    },
}
pkl_path = os.path.join(SAVE_DIR, 'two_strategies_full.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(pkl_data, f)
print("Pickle 已保存: {}".format(pkl_path))

print("\n" + "=" * 70)
print("全部完成！")
print("=" * 70)

In [ ]:
"""
两策略结果可视化
适配 temp_offset_two_strategies_v3.py 的输出
按 # %% 分cell在Jupyter中运行
"""

# %% Cell 1 - 加载数据

import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os
from oggm import cfg

SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'offset_two_strategies')

# 加载结果
df_all = pd.read_csv(os.path.join(SAVE_DIR, 'two_strategies_results.csv'))
with open(os.path.join(SAVE_DIR, 'two_strategies_full.pkl'), 'rb') as f:
    pkl = pickle.load(f)

eq_offsets = pkl['eq_offsets']
spinup_ok_ids = pkl['spinup_ok_ids']

df_a = df_all[df_all['strategy'] == 'A_keep_offset'].copy()
df_b = df_all[df_all['strategy'] == 'B_remove_offset'].copy()

print("策略A: {} 条, 策略B: {} 条".format(len(df_a), len(df_b)))
print("spinup成功: {} 条".format(len(spinup_ok_ids)))

# %% Cell 2 - 辅助函数：读取 spinup + 策略A/B 正演时间序列

Y0 = 1500
SPINUP_YEARS = 300
SUFFIX_SPINUP = '_spinup_perglacier_offset'
SUFFIX_FW_A = '_forward_keep_offset'
SUFFIX_FW_B = '_forward_remove_offset'

# 需要 gdirs 对象来读文件（假设 gdirs_lia 在环境中）
gdir_dict = {g.rgi_id: g for g in gdirs_lia if g.rgi_id in spinup_ok_ids}


def load_spinup_ts(gdir):
    """读取 spin-up 时间序列"""
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_SPINUP)
        if not os.path.exists(fp):
            return None
        ds = xr.open_dataset(fp)
        n = len(ds['area_m2'].values)
        years = np.linspace(Y0 - SPINUP_YEARS, Y0, n)
        result = {
            'years': years,
            'area_km2': ds['area_m2'].values / 1e6,
            'volume_km3': ds['volume_m3'].values / 1e9,
            'length_km': ds['length_m'].values / 1e3,
        }
        ds.close()
        return result
    except:
        return None


def load_forward_ts(gdir, suffix):
    """读取正演时间序列"""
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix)
        if not os.path.exists(fp):
            return None
        ds = xr.open_dataset(fp)
        n = len(ds['area_m2'].values)
        result = {
            'years': np.arange(Y0, Y0 + n),
            'area_km2': ds['area_m2'].values / 1e6,
            'volume_km3': ds['volume_m3'].values / 1e9,
            'length_km': ds['length_m'].values / 1e3,
        }
        ds.close()
        return result
    except:
        return None


def splice_ts(sp, fw):
    """拼接 spinup + forward，去掉重复点"""
    if sp is None and fw is None:
        return None
    result = {}
    for var in ['area_km2', 'volume_km3', 'length_km']:
        parts = []
        if sp is not None:
            parts.append(sp[var])
        if fw is not None:
            parts.append(fw[var][1:] if sp is not None else fw[var])
        result[var] = np.concatenate(parts)

    parts_y = []
    if sp is not None:
        parts_y.append(sp['years'])
    if fw is not None:
        parts_y.append(fw['years'][1:] if sp is not None else fw['years'])
    result['years'] = np.concatenate(parts_y)
    return result


# %% Cell 3 - 图1: 逐冰川 spinup+正演 面积时间序列（策略A vs B）

# 取有结果的冰川
common_uids = sorted(set(df_a['uid'].tolist()) & set(df_b['uid'].tolist()))
n_glaciers = len(common_uids)
ncols = 5
nrows = int(np.ceil(n_glaciers / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
axes = axes.flatten()

for idx, uid in enumerate(common_uids):
    ax = axes[idx]

    row_a = df_a[df_a['uid'] == uid].iloc[0]
    row_b = df_b[df_b['uid'] == uid].iloc[0]
    rgi_id = row_a['rgi_id']
    gdir = gdir_dict.get(rgi_id)
    if gdir is None:
        continue

    # 读时间序列
    sp = load_spinup_ts(gdir)
    fw_a = load_forward_ts(gdir, SUFFIX_FW_A)
    fw_b = load_forward_ts(gdir, SUFFIX_FW_B)

    ts_a = splice_ts(sp, fw_a)
    ts_b = splice_ts(sp, fw_b)

    # 画 spinup（灰色）
    if sp is not None:
        ax.plot(sp['years'], sp['area_km2'], color='gray', linewidth=1.5,
                alpha=0.6, label='spin-up')

    # 策略A（红）
    if fw_a is not None:
        ax.plot(fw_a['years'], fw_a['area_km2'], color='red', linewidth=2.0,
                alpha=0.9, label='A: keep offset')

    # 策略B（蓝）
    if fw_b is not None:
        ax.plot(fw_b['years'], fw_b['area_km2'], color='blue', linewidth=2.0,
                alpha=0.9, label='B: remove offset')

    # 分界线
    ax.axvline(Y0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

    # LIA参考线
    ax.axhline(row_a['lia_init_area'], color='green', linestyle=':', linewidth=1.5)

    # RGI参考线
    ax.axhline(row_a['rgi_area'], color='orange', linestyle='--', linewidth=1.8)

    ax.set_title('{} | off={:+.2f}K\nA={:.3f} B={:.3f}'.format(
        uid, eq_offsets.get(rgi_id, 0),
        row_a['sim_rgi_ratio'], row_b['sim_rgi_ratio']),
        fontsize=9)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

    if idx == 0:
        ax.legend(fontsize=6, loc='upper right')

# 隐藏空白子图
for i in range(n_glaciers, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('Spinup + Forward: Strategy A (red, keep offset) vs B (blue, remove offset)\n'
             'Green:=LIA init | Orange--=RGI observed | y0={}'.format(Y0),
             fontsize=13, fontweight='bold')
fig.supxlabel('Year', fontsize=12)
fig.supylabel('Area (km²)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig1_per_glacier_timeseries.png'), dpi=150, bbox_inches='tight')
plt.show()
print("图1已保存")


# %% Cell 4 - 图2: ratio对比柱状图

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# 上图：A vs B ratio 对比
merged = df_a[['uid', 'sim_rgi_ratio', 'rgi_area']].rename(
    columns={'sim_rgi_ratio': 'ratio_A'}).merge(
    df_b[['uid', 'sim_rgi_ratio']].rename(
    columns={'sim_rgi_ratio': 'ratio_B'}), on='uid')
merged = merged.sort_values('ratio_B').reset_index(drop=True)

x = np.arange(len(merged))
width = 0.35

bars_a = ax1.bar(x - width/2, merged['ratio_A'], width, color='red', alpha=0.7,
                  edgecolor='black', linewidth=0.5, label='A: keep offset')
bars_b = ax1.bar(x + width/2, merged['ratio_B'], width, color='blue', alpha=0.7,
                  edgecolor='black', linewidth=0.5, label='B: remove offset')

ax1.axhline(1.0, color='black', linestyle='--', linewidth=1.5)
ax1.fill_between([-0.5, len(merged)-0.5], 0.8, 1.2, alpha=0.1, color='green')
ax1.set_xticks(x)
ax1.set_xticklabels(merged['uid'], rotation=45, fontsize=9)
ax1.set_ylabel('sim_2000 / RGI area')
ax1.set_title('Strategy A vs B: sim/RGI ratio per glacier (y0={})'.format(Y0))
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# 下图：LIA / spinup_end / A_end / B_end / RGI 面积对比
merged2 = df_a[['uid', 'lia_init_area', 'spinup_end_area', 'forward_end_area', 'rgi_area']].rename(
    columns={'forward_end_area': 'A_end'}).merge(
    df_b[['uid', 'forward_end_area']].rename(
    columns={'forward_end_area': 'B_end'}), on='uid')
merged2 = merged2.sort_values('rgi_area').reset_index(drop=True)

x2 = np.arange(len(merged2))
w = 0.18
ax2.bar(x2 - 2*w, merged2['lia_init_area'], w, color='green', alpha=0.7, label='LIA init')
ax2.bar(x2 - w,   merged2['spinup_end_area'], w, color='gray', alpha=0.7, label='Spinup end')
ax2.bar(x2,        merged2['A_end'], w, color='red', alpha=0.7, label='A: sim 2000')
ax2.bar(x2 + w,    merged2['B_end'], w, color='blue', alpha=0.7, label='B: sim 2000')
ax2.bar(x2 + 2*w,  merged2['rgi_area'], w, color='orange', alpha=0.7, label='RGI observed')

ax2.set_xticks(x2)
ax2.set_xticklabels(merged2['uid'], rotation=45, fontsize=9)
ax2.set_ylabel('Area (km²)')
ax2.set_title('Area comparison: LIA → Spinup → Sim2000 → RGI')
ax2.legend(fontsize=8, ncol=5)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig2_ratio_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("图2已保存")


# %% Cell 5 - 图3: offset vs ratio 散点图

fig, ax = plt.subplots(figsize=(10, 6))

for label, df_x, color, marker in [
    ('A: keep offset', df_a, 'red', 'o'),
    ('B: remove offset', df_b, 'blue', 's')]:
    ax.scatter(df_x['offset'], df_x['sim_rgi_ratio'],
               c=color, marker=marker, alpha=0.7, s=80, edgecolors='black',
               linewidth=0.5, label=label)

    # 标注uid
    for _, row in df_x.iterrows():
        ax.annotate(row['uid'], (row['offset'], row['sim_rgi_ratio']),
                    fontsize=6, alpha=0.6, ha='left',
                    xytext=(3, 3), textcoords='offset points')

ax.axhline(1.0, color='black', linestyle='--', linewidth=1.5)
ax.fill_between(ax.get_xlim(), 0.8, 1.2, alpha=0.1, color='green')
ax.set_xlabel('Per-glacier eq_offset (K)')
ax.set_ylabel('sim_2000 / RGI area')
ax.set_title('Offset vs Ratio: Strategy A (red) vs B (blue)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig3_offset_vs_ratio.png'), dpi=150, bbox_inches='tight')
plt.show()
print("图3已保存")


# %% Cell 6 - 图4: 代表性冰川（ratio最接近1.0）三合一

# 找策略B中ratio最接近1.0的冰川
best_row = df_b.loc[(df_b['sim_rgi_ratio'] - 1.0).abs().idxmin()]
best_rgi_id = best_row['rgi_id']
best_uid = best_row['uid']
best_gdir = gdir_dict.get(best_rgi_id)

print("展示冰川: {} (B_ratio={:.3f})".format(best_uid, best_row['sim_rgi_ratio']))

sp = load_spinup_ts(best_gdir)
fw_a = load_forward_ts(best_gdir, SUFFIX_FW_A)
fw_b = load_forward_ts(best_gdir, SUFFIX_FW_B)

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for fw, color, lbl in [(fw_a, 'red', 'A: keep offset'), (fw_b, 'blue', 'B: remove offset')]:
    if fw is None:
        continue
    for i, var in enumerate(['area_km2', 'volume_km3', 'length_km']):
        # spinup
        if sp is not None:
            axes[i].plot(sp['years'], sp[var], color='gray', linewidth=1.5, alpha=0.5,
                         label='spin-up' if color == 'red' else '')
        # forward
        axes[i].plot(fw['years'], fw[var], color=color, linewidth=2.0, alpha=0.9, label=lbl)

# 分界线
for ax in axes:
    ax.axvline(Y0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.grid(True, alpha=0.3)

# 面积参考线
row_info = df_a[df_a['rgi_id'] == best_rgi_id].iloc[0]
axes[0].axhline(row_info['lia_init_area'], color='green', linestyle=':', linewidth=2, label='LIA init')
axes[0].axhline(row_info['rgi_area'], color='orange', linestyle='--', linewidth=2, label='RGI observed')

axes[0].set_ylabel('Area (km²)')
axes[0].set_title('{} | offset={:+.2f}K | A_ratio={:.3f}, B_ratio={:.3f}'.format(
    best_uid, eq_offsets.get(best_rgi_id, 0),
    row_info['sim_rgi_ratio'],
    best_row['sim_rgi_ratio']))
axes[0].legend(fontsize=8, ncol=3)

axes[1].set_ylabel('Volume (km³)')
axes[2].set_ylabel('Length (km)')
axes[2].set_xlabel('Year')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'fig4_best_glacier_detail.png'), dpi=150, bbox_inches='tight')
plt.show()
print("图4已保存")


# %% Cell 7 - 汇总打印

print("\n" + "=" * 100)
print("汇总 (y0={})".format(Y0))
print("=" * 100)

for label, df_x in [('A (keep offset)', df_a), ('B (remove offset)', df_b)]:
    ratios = df_x['sim_rgi_ratio'].dropna()
    n_good = ((ratios >= 0.8) & (ratios <= 1.2)).sum()
    print("\n策略 {}:".format(label))
    print("  N={}, 均值={:.3f}, 中位数={:.3f}, std={:.3f}".format(
        len(ratios), ratios.mean(), ratios.median(), ratios.std()))
    print("  [0.8-1.2]: {}/{} = {:.1f}%".format(
        n_good, len(ratios), n_good / len(ratios) * 100 if len(ratios) > 0 else 0))
    print("  min={:.3f}, max={:.3f}".format(ratios.min(), ratios.max()))

In [ ]:
# 补充RGI冰川的反演和flowline初始化
from oggm.workflow import execute_entity_task

cfg.PATHS['working_dir'] = RGI_WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH

execute_entity_task(tasks.apparent_mb_from_any_mb, gdirs_rgi)
execute_entity_task(tasks.prepare_for_inversion, gdirs_rgi)
execute_entity_task(tasks.mass_conservation_inversion, gdirs_rgi)
execute_entity_task(tasks.filter_inversion_output, gdirs_rgi)
execute_entity_task(tasks.init_present_time_glacier, gdirs_rgi)

# 验证
print(os.path.exists(os.path.join(gdirs_rgi[0].dir, 'model_flowlines.pkl')))

In [ ]:
"""
闭环测试：RGI冰川 变冷增长 → 变暖退缩 → 能否回到原始面积？
=================================================================
目的：验证OGGM模型框架本身能否复现"增长→退缩"过程
方法：
  Phase 0: 记录RGI冰川的现代面积（基准）
  Phase 1: 人为降温，run_constant_climate让冰川增长（模拟LIA增长）
  Phase 2: 去掉降温，run_constant_climate看冰川能否退回原始面积

  如果退回 → 模型没问题，问题在重建气候数据
  如果退不回 → 模型本身有结构性问题

使用CRU气候 + 现代校准参数，不涉及重建气候和LIA轮廓
"""

# %% Cell 1 - 配置

import json, os, gc, time
import numpy as np
import xarray as xr
import pandas as pd
import pickle

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ---------- 实验参数 ----------
TEST_N = 20                      # 测试冰川数量
COOL_OFFSETS = [-1.0, -1.5, -2.0]  # 降温量（测试多个）
GROW_YEARS = 300                  # 变冷增长阶段年数
SHRINK_YEARS = 500                # 变暖退缩阶段年数
Y0_MODERN = 1990                  # 现代气候参考年（CRU数据范围内）
HALFSIZE = 15                     # 气候窗口半宽

# ---------- 多进程配置 ----------
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 8
cfg.PARAMS['continue_on_error'] = True

# ---------- 工作目录 ----------
# 切换回RGI工作目录
RGI_WORKING_DIR = "/mnt/f/oggm_related/Rec_climate/Rec_rgi_calibration"
DEM_PATH = "/mnt/d/01 文件整理/04 数据/HMA_newregion/DEM_clip/Himalaya_3regions_DEM.tif"

cfg.PATHS['working_dir'] = RGI_WORKING_DIR
cfg.PATHS['dem_file'] = DEM_PATH

SAVE_DIR = os.path.join(RGI_WORKING_DIR, 'closed_loop_test')
os.makedirs(SAVE_DIR, exist_ok=True)

print("=" * 70)
print("闭环测试：RGI冰川 变冷增长 → 变暖退缩")
print("=" * 70)
print("降温量: {}".format(COOL_OFFSETS))
print("增长阶段: {}年, 退缩阶段: {}年".format(GROW_YEARS, SHRINK_YEARS))
print("现代气候参考: y0={}, halfsize={}".format(Y0_MODERN, HALFSIZE))

# %% Cell 2 - 筛选测试冰川 & 记录基准面积

print("\n[Step 0] 筛选冰川 & 记录基准面积...")

# 筛选有完整flowline的冰川
test_gdirs = [g for g in gdirs_rgi
              if os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl'))
              and os.path.exists(os.path.join(g.dir, 'mb_calib.json'))][:TEST_N]

print("测试冰川: {} 条".format(len(test_gdirs)))

# 记录基准面积和原始参数
baseline_info = {}
for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    rgi_area_km2 = gdir.rgi_area_m2 / 1e6

    with open(os.path.join(gdir.dir, 'mb_calib.json'), 'r') as f:
        p = json.load(f)

    baseline_info[rgi_id] = {
        'rgi_area_km2': rgi_area_km2,
        'melt_f': p['melt_f'],
        'temp_bias': p['temp_bias'],
        'prcp_fac': p['prcp_fac'],
    }
    uid = rgi_id.split('.')[-1]
    print("  {} : area={:.2f} km²  melt_f={:.2f}  temp_bias={:.2f}".format(
        uid, rgi_area_km2, p['melt_f'], p['temp_bias']))

# %% Cell 3 - 辅助函数

def set_temp_bias(gdirs, offset):
    """给所有冰川的temp_bias加上offset"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        orig_tb = baseline_info[rgi_id]['temp_bias']
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = orig_tb + offset
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

def restore_params(gdirs):
    """恢复原始参数"""
    for gdir in gdirs:
        rgi_id = gdir.rgi_id
        calib_path = os.path.join(gdir.dir, 'mb_calib.json')
        with open(calib_path, 'r') as f:
            p = json.load(f)
        p['temp_bias'] = baseline_info[rgi_id]['temp_bias']
        p['melt_f'] = baseline_info[rgi_id]['melt_f']
        p['prcp_fac'] = baseline_info[rgi_id]['prcp_fac']
        with open(calib_path, 'w') as f:
            json.dump(p, f, indent=2)

# %% Cell 4 - Phase 1: 变冷增长 + Phase 2: 变暖退缩

all_results = []

for cool_offset in COOL_OFFSETS:
    print("\n" + "=" * 70)
    print("降温量: {:+.1f}K".format(cool_offset))
    print("=" * 70)

    suffix_grow = '_grow_cool{}'.format(
        str(cool_offset).replace('.', 'p').replace('-', 'n'))
    suffix_shrink = '_shrink_cool{}'.format(
        str(cool_offset).replace('.', 'p').replace('-', 'n'))

    # ========== Phase 1: 变冷增长 ==========
    print("\n  [Phase 1] 变冷增长 ({}年, offset={:+.1f}K)...".format(
        GROW_YEARS, cool_offset))

    set_temp_bias(test_gdirs, cool_offset)

    t0 = time.time()
    try:
        execute_entity_task(
            tasks.run_constant_climate, test_gdirs,
            nyears=GROW_YEARS, y0=Y0_MODERN, halfsize=HALFSIZE,
            climate_filename='climate_historical',
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            output_filesuffix=suffix_grow,
            store_model_geometry=True,
        )
    except Exception as e:
        print("    Phase1 异常: {}".format(e))
    print("    耗时: {:.1f}min".format((time.time() - t0) / 60))

    # 检查成功的冰川
    grow_ok = []
    for gdir in test_gdirs:
        try:
            fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_grow)
            if os.path.exists(fp):
                grow_ok.append(gdir)
        except:
            pass
    print("    增长成功: {}/{}".format(len(grow_ok), len(test_gdirs)))

    # 打印增长结果
    print("\n    {:<10} {:>10} {:>10} {:>8}".format(
        "UID", "orig_km2", "grown_km2", "chg%"))
    print("    " + "-" * 45)
    for gdir in grow_ok:
        uid = gdir.rgi_id.split('.')[-1]
        try:
            fp = gdir.get_filepath('model_diagnostics', filesuffix=suffix_grow)
            ds = xr.open_dataset(fp)
            a0 = float(ds['area_m2'].values[0]) / 1e6
            ae = float(ds['area_m2'].values[-1]) / 1e6
            pct = (ae - a0) / a0 * 100
            ds.close()
            print("    {:<10} {:>10.2f} {:>10.2f} {:>+8.1f}%".format(uid, a0, ae, pct))
        except:
            pass

    # ========== Phase 2: 变暖退缩（去掉降温） ==========
    print("\n  [Phase 2] 变暖退缩 ({}年, 恢复原始参数)...".format(SHRINK_YEARS))

    restore_params(grow_ok)

    t0 = time.time()
    try:
        execute_entity_task(
            tasks.run_constant_climate, grow_ok,
            nyears=SHRINK_YEARS, y0=Y0_MODERN, halfsize=HALFSIZE,
            climate_filename='climate_historical',
            mb_model_class=MyMBModel,
            evolution_model=FluxBasedModel,
            init_model_filesuffix=suffix_grow,
            output_filesuffix=suffix_shrink,
            store_model_geometry=True,
        )
    except Exception as e:
        print("    Phase2 异常: {}".format(e))
    print("    耗时: {:.1f}min".format((time.time() - t0) / 60))

    # ========== 提取结果 ==========
    for gdir in grow_ok:
        rgi_id = gdir.rgi_id
        uid = rgi_id.split('.')[-1]
        try:
            # 增长阶段
            fp1 = gdir.get_filepath('model_diagnostics', filesuffix=suffix_grow)
            ds1 = xr.open_dataset(fp1)
            orig_area = float(ds1['area_m2'].values[0]) / 1e6
            grown_area = float(ds1['area_m2'].values[-1]) / 1e6
            ds1.close()

            # 退缩阶段
            fp2 = gdir.get_filepath('model_diagnostics', filesuffix=suffix_shrink)
            if not os.path.exists(fp2):
                continue
            ds2 = xr.open_dataset(fp2)
            shrunk_area = float(ds2['area_m2'].values[-1]) / 1e6
            ds2.close()

            rgi_area = baseline_info[rgi_id]['rgi_area_km2']
            all_results.append({
                'uid': uid,
                'rgi_id': rgi_id,
                'cool_offset': cool_offset,
                'rgi_area': rgi_area,
                'orig_area': orig_area,
                'grown_area': grown_area,
                'grow_pct': (grown_area - orig_area) / orig_area * 100,
                'shrunk_area': shrunk_area,
                'shrink_pct': (shrunk_area - grown_area) / grown_area * 100,
                'final_vs_orig': shrunk_area / rgi_area,
                'recovered': abs(shrunk_area - rgi_area) / rgi_area < 0.1,
            })
        except Exception as e:
            print("    提取失败 {}: {}".format(uid, e))

    # 恢复参数
    restore_params(test_gdirs)
    gc.collect()

# %% Cell 5 - 结果分析

print("\n" + "=" * 100)
print("[结果分析]")
print("=" * 100)

df = pd.DataFrame(all_results)

for cool in COOL_OFFSETS:
    sub = df[df['cool_offset'] == cool]
    if len(sub) == 0:
        continue

    print("\n--- 降温 {:+.1f}K ---".format(cool))
    print("{:<10} {:>8} {:>10} {:>10} {:>10} {:>10} {:>8}".format(
        "UID", "RGI", "grown", "shrunk", "final/RGI", "grow%", "recov?"))
    print("-" * 72)

    for _, row in sub.iterrows():
        print("{:<10} {:>8.2f} {:>10.2f} {:>10.2f} {:>10.3f} {:>+10.1f}% {:>8}".format(
            row['uid'], row['rgi_area'], row['grown_area'],
            row['shrunk_area'], row['final_vs_orig'],
            row['grow_pct'],
            'YES' if row['recovered'] else 'NO'))

    n_recovered = sub['recovered'].sum()
    print("\n  恢复率: {}/{} = {:.1f}%".format(
        n_recovered, len(sub), n_recovered / len(sub) * 100))
    print("  final/RGI: 均值={:.3f}  中位数={:.3f}  std={:.3f}".format(
        sub['final_vs_orig'].mean(),
        sub['final_vs_orig'].median(),
        sub['final_vs_orig'].std()))

# %% Cell 6 - 可视化

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(COOL_OFFSETS), figsize=(7 * len(COOL_OFFSETS), 6))
if len(COOL_OFFSETS) == 1:
    axes = [axes]

for ax, cool in zip(axes, COOL_OFFSETS):
    sub = df[df['cool_offset'] == cool].sort_values('final_vs_orig')
    if len(sub) == 0:
        continue

    x = np.arange(len(sub))
    width = 0.2

    ax.bar(x - 1.5*width, sub['rgi_area'], width, color='orange', alpha=0.8, label='RGI orig')
    ax.bar(x - 0.5*width, sub['grown_area'], width, color='blue', alpha=0.8, label='Grown (cool)')
    ax.bar(x + 0.5*width, sub['shrunk_area'], width, color='red', alpha=0.8, label='Shrunk (warm)')

    ax.set_xticks(x)
    ax.set_xticklabels(sub['uid'], rotation=45, fontsize=8)
    ax.set_ylabel('Area (km²)')
    ax.set_title('Cool offset = {:+.1f}K\nRecovered: {}/{}'.format(
        cool, sub['recovered'].sum(), len(sub)))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Closed-loop test: Cool → Grow → Warm → Shrink → Compare to original',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# %% Cell 7 - 逐冰川时间序列（增长+退缩拼接）

n_glaciers = min(len(test_gdirs), 20)
ncols = 5
nrows = int(np.ceil(n_glaciers / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
axes = axes.flatten()

# 用中间的cool_offset画图
plot_cool = COOL_OFFSETS[len(COOL_OFFSETS) // 2]
suffix_grow = '_grow_cool{}'.format(
    str(plot_cool).replace('.', 'p').replace('-', 'n'))
suffix_shrink = '_shrink_cool{}'.format(
    str(plot_cool).replace('.', 'p').replace('-', 'n'))

plot_idx = 0
for gdir in test_gdirs:
    if plot_idx >= n_glaciers:
        break
    rgi_id = gdir.rgi_id
    uid = rgi_id.split('.')[-1]
    ax = axes[plot_idx]

    try:
        # 增长阶段
        fp1 = gdir.get_filepath('model_diagnostics', filesuffix=suffix_grow)
        ds1 = xr.open_dataset(fp1)
        n1 = len(ds1['area_m2'].values)
        years1 = np.arange(0, n1)
        area1 = ds1['area_m2'].values / 1e6
        ds1.close()

        # 退缩阶段
        fp2 = gdir.get_filepath('model_diagnostics', filesuffix=suffix_shrink)
        ds2 = xr.open_dataset(fp2)
        n2 = len(ds2['area_m2'].values)
        years2 = np.arange(n1 - 1, n1 - 1 + n2)
        area2 = ds2['area_m2'].values / 1e6
        ds2.close()

        ax.plot(years1, area1, color='blue', linewidth=2, label='Cool (grow)')
        ax.plot(years2, area2, color='red', linewidth=2, label='Warm (shrink)')
        ax.axvline(n1 - 1, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

        rgi_area = baseline_info[rgi_id]['rgi_area_km2']
        ax.axhline(rgi_area, color='orange', linestyle='--', linewidth=1.8, label='RGI orig')

        final_ratio = area2[-1] / rgi_area
        ax.set_title('{} | cool={:+.1f}K\nfinal/RGI={:.3f}'.format(
            uid, plot_cool, final_ratio), fontsize=9)

    except Exception as e:
        ax.set_title('{} | FAILED'.format(uid), fontsize=9)

    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)
    if plot_idx == 0:
        ax.legend(fontsize=6)
    plot_idx += 1

for i in range(plot_idx, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('Closed-loop: Cool {:+.1f}K ({}yr) → Warm ({}yr) | Orange--=RGI original'.format(
    plot_cool, GROW_YEARS, SHRINK_YEARS),
    fontsize=13, fontweight='bold')
fig.supxlabel('Year (simulation time)', fontsize=12)
fig.supylabel('Area (km²)', fontsize=12)
plt.tight_layout()
plt.show()

# %% Cell 8 - 保存

csv_path = os.path.join(SAVE_DIR, 'closed_loop_results.csv')
df.to_csv(csv_path, index=False)
print("CSV 已保存: {}".format(csv_path))

pkl_path = os.path.join(SAVE_DIR, 'closed_loop_full.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump({
        'results': all_results,
        'baseline_info': baseline_info,
        'test_gdirs_ids': [g.rgi_id for g in test_gdirs],
        'config': {
            'COOL_OFFSETS': COOL_OFFSETS,
            'GROW_YEARS': GROW_YEARS,
            'SHRINK_YEARS': SHRINK_YEARS,
            'Y0_MODERN': Y0_MODERN,
            'HALFSIZE': HALFSIZE,
        },
    }, f)
print("Pickle 已保存: {}".format(pkl_path))

print("\n" + "=" * 70)
print("闭环测试完成！")
print("=" * 70)

In [ ]:
"""
LIA几何上MB≈0校准 + spin-up + 正演
====================================
思路：
  Step 1: 在LIA冰川几何上，用重建气候(Y0±30)，以MB≈0为目标
          校准 melt_f 和 temp_bias（prcp_fac=8不变）
  Step 2: 用校准后的参数做 spin-up（不需要offset）
  Step 3: 正演到2000年，参数不变，让气候变化驱动退缩
  Step 4: 对比sim_2000面积 vs RGI观测面积

关键改进：参数和几何、气候自洽，不再需要temp_offset补偿
"""

# %% Cell 1 - 配置

import json, os, gc, time
import numpy as np
import xarray as xr
import pandas as pd
import pickle
from scipy.optimize import brentq

from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

class MyMBModel(MonthlyTIModel):
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# ---------- 实验参数 ----------
TEST_N = 20
Y0 = 1500
HALFSIZE = 30
SPINUP_YEARS = 300
CLIMATE_SUFFIX = '_ensemble_rec_ref'

# melt_f 搜索范围
MELT_F_MIN = 0.5
MELT_F_MAX = 20.0

# temp_bias 搜索范围（在原始值附近搜索）
TB_SEARCH_RANGE = 5.0  # ± 5K

# ---------- 多进程 ----------
cfg.PARAMS['use_multiprocessing'] = True
cfg.PARAMS['mp_processes'] = 15
cfg.PARAMS['continue_on_error'] = True

# ---------- 保存目录 ----------
SAVE_DIR = os.path.join(cfg.PATHS['working_dir'], 'lia_mb0_calibration')
os.makedirs(SAVE_DIR, exist_ok=True)

print("=" * 70)
print("LIA几何 MB≈0 校准 + spin-up + 正演")
print("=" * 70)

# %% Cell 2 - 重建派生变量（同之前）

rgi_area_map = {}
rgi_id_map = {}
original_params = {}

for gdir in gdirs_lia:
    uid = int(gdir.rgi_id.split('.')[-1])
    try:
        main_rgi_id = lia_to_main_rgi[uid]
        rgi_gdir = [g for g in gdirs_rgi if g.rgi_id == main_rgi_id][0]
        rgi_area_map[gdir.rgi_id] = rgi_gdir.rgi_area_m2 / 1e6
        rgi_id_map[gdir.rgi_id] = main_rgi_id
    except:
        continue
    try:
        params = lia_mb_params_cru[uid]
        original_params[gdir.rgi_id] = {
            'temp_bias': params['temp_bias'],
            'melt_f':    params['melt_f'],
            'prcp_fac':  params['prcp_fac'],
        }
    except:
        pass

print("rgi_area_map : {}".format(len(rgi_area_map)))
print("original_params: {}".format(len(original_params)))

# %% Cell 3 - 筛选冰川

test_gdirs = [g for g in gdirs_lia
              if g.rgi_id in original_params
              and g.rgi_id in rgi_area_map
              and os.path.exists(os.path.join(g.dir, 'model_flowlines.pkl'))][:TEST_N]

print("测试冰川: {} 条".format(len(test_gdirs)))

# %% Cell 4 - 在LIA几何上以MB≈0校准 melt_f

print("\n[Step 1] 在LIA几何上校准 melt_f (目标MB≈0)...")
print("气候: Y0={}, halfsize={}, suffix={}".format(Y0, HALFSIZE, CLIMATE_SUFFIX))
print("melt_f范围: [{}, {}]".format(MELT_F_MIN, MELT_F_MAX))

t_start = time.time()
lia_calib_params = {}

for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    uid = rgi_id.split('.')[-1]
    orig = original_params[rgi_id]
    fls = gdir.read_pickle('model_flowlines')

    def mb_for_meltf(melt_f):
        """给定melt_f，计算LIA几何上Y0±30的平均MB"""
        try:
            mb_model = MyMBModel(
                gdir, filename='gcm_data',
                input_filesuffix=CLIMATE_SUFFIX,
                melt_f=melt_f,
                temp_bias=orig['temp_bias'],
                prcp_fac=orig['prcp_fac'],
            )
            avg = np.mean([mb_model.get_specific_mb(fls=fls, year=yr)
                           for yr in range(Y0 - HALFSIZE, Y0 + HALFSIZE + 1)])
            return avg
        except:
            return np.nan

    # 先检查搜索范围两端的MB符号
    mb_lo = mb_for_meltf(MELT_F_MIN)
    mb_hi = mb_for_meltf(MELT_F_MAX)

    if np.isnan(mb_lo) or np.isnan(mb_hi):
        print("  {} : MB计算失败 (lo={}, hi={})".format(uid, mb_lo, mb_hi))
        continue

    if mb_lo * mb_hi > 0:
        # 同号，没有零点
        print("  {} : 无零点 melt_f=[{:.1f},{:.1f}] → MB=[{:.0f},{:.0f}]".format(
            uid, MELT_F_MIN, MELT_F_MAX, mb_lo, mb_hi))
        # 取MB绝对值最小的端点作为fallback
        if abs(mb_lo) < abs(mb_hi):
            lia_calib_params[rgi_id] = {
                'melt_f': MELT_F_MIN,
                'temp_bias': orig['temp_bias'],
                'prcp_fac': orig['prcp_fac'],
                'calib_mb': mb_lo,
                'status': 'fallback_lo',
            }
        else:
            lia_calib_params[rgi_id] = {
                'melt_f': MELT_F_MAX,
                'temp_bias': orig['temp_bias'],
                'prcp_fac': orig['prcp_fac'],
                'calib_mb': mb_hi,
                'status': 'fallback_hi',
            }
        continue

    # Brent法找零点
    try:
        best_mf = brentq(mb_for_meltf, MELT_F_MIN, MELT_F_MAX, xtol=0.01)
        calib_mb = mb_for_meltf(best_mf)
        lia_calib_params[rgi_id] = {
            'melt_f': best_mf,
            'temp_bias': orig['temp_bias'],
            'prcp_fac': orig['prcp_fac'],
            'calib_mb': calib_mb,
            'status': 'success',
        }
        print("  {} : melt_f={:.2f} (orig={:.2f}) MB={:.0f} ✓".format(
            uid, best_mf, orig['melt_f'], calib_mb))
    except Exception as e:
        print("  {} : brentq失败 - {}".format(uid, e))

print("\n校准完成: {}/{}, 耗时: {:.1f}min".format(
    len(lia_calib_params), len(test_gdirs), (time.time() - t_start) / 60))

# 打印校准结果汇总
print("\n{:<8} {:>10} {:>10} {:>10} {:>8} {:>8}".format(
    "UID", "orig_mf", "calib_mf", "ratio", "MB", "status"))
print("-" * 60)
for rgi_id, cp in lia_calib_params.items():
    uid = rgi_id.split('.')[-1]
    orig_mf = original_params[rgi_id]['melt_f']
    print("{:<8} {:>10.2f} {:>10.2f} {:>10.2f} {:>8.0f} {:>8}".format(
        uid, orig_mf, cp['melt_f'], cp['melt_f'] / orig_mf,
        cp['calib_mb'], cp['status']))

# 保存checkpoint
pkl_calib = os.path.join(SAVE_DIR, 'lia_calib_params.pkl')
with open(pkl_calib, 'wb') as f:
    pickle.dump(lia_calib_params, f)
print("\n校准参数已保存: {}".format(pkl_calib))

# %% Cell 5 - 写入校准参数到 mb_calib.json

print("\n[Step 1.5] 写入校准参数...")

calib_gdirs = [g for g in test_gdirs if g.rgi_id in lia_calib_params]
print("有校准参数的冰川: {} 条".format(len(calib_gdirs)))

for gdir in calib_gdirs:
    rgi_id = gdir.rgi_id
    cp = lia_calib_params[rgi_id]
    calib_path = os.path.join(gdir.dir, 'mb_calib.json')
    with open(calib_path, 'r') as f:
        p = json.load(f)
    p['melt_f'] = cp['melt_f']
    p['temp_bias'] = cp['temp_bias']
    p['prcp_fac'] = cp['prcp_fac']
    with open(calib_path, 'w') as f:
        json.dump(p, f, indent=2)

print("写入完成")

# %% Cell 6 - Spin-up（不需要offset）

SUFFIX_SPINUP = '_spinup_mb0calib'

print("\n[Step 2] Spin-up ({}年, y0={}, 无offset)...".format(SPINUP_YEARS, Y0))

t0 = time.time()
try:
    execute_entity_task(
        tasks.run_constant_climate, calib_gdirs,
        nyears=SPINUP_YEARS, y0=Y0, halfsize=HALFSIZE,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=SUFFIX_SPINUP,
        store_model_geometry=True,
    )
except Exception as e:
    print("  Spin-up异常: {}".format(e))
print("  Spin-up耗时: {:.1f}min".format((time.time() - t0) / 60))

spinup_ok = []
for gdir in calib_gdirs:
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_SPINUP)
        if os.path.exists(fp):
            spinup_ok.append(gdir)
    except:
        pass
print("  Spin-up成功: {}/{}".format(len(spinup_ok), len(calib_gdirs)))

# 打印spin-up面积变化
print("\n  {:<8} {:>10} {:>10} {:>8} {:>8}".format(
    "UID", "init_km2", "end_km2", "chg%", "melt_f"))
print("  " + "-" * 50)
for gdir in spinup_ok:
    uid = gdir.rgi_id.split('.')[-1]
    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_SPINUP)
        ds = xr.open_dataset(fp)
        a0 = float(ds['area_m2'].values[0]) / 1e6
        ae = float(ds['area_m2'].values[-1]) / 1e6
        pct = (ae - a0) / a0 * 100
        ds.close()
        mf = lia_calib_params[gdir.rgi_id]['melt_f']
        print("  {:<8} {:>10.2f} {:>10.2f} {:>+8.1f}% {:>8.2f}".format(
            uid, a0, ae, pct, mf))
    except:
        pass

# %% Cell 7 - 正演到2000年

SUFFIX_FORWARD = '_forward_mb0calib'

print("\n[Step 3] 正演到2000年 ({} 条冰川)...".format(len(spinup_ok)))

t0 = time.time()
try:
    execute_entity_task(
        tasks.run_from_climate_data, spinup_ok,
        min_ys=Y0, ye=2000,
        climate_filename='gcm_data',
        climate_input_filesuffix=CLIMATE_SUFFIX,
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        init_model_filesuffix=SUFFIX_SPINUP,
        output_filesuffix=SUFFIX_FORWARD,
    )
except Exception as e:
    print("  正演异常: {}".format(e))
print("  正演耗时: {:.1f}min".format((time.time() - t0) / 60))

# %% Cell 8 - 提取结果

print("\n[Step 4] 提取结果...")

results = []
for gdir in spinup_ok:
    rgi_id = gdir.rgi_id
    uid = rgi_id.split('.')[-1]
    try:
        # 正演结果
        fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_FORWARD)
        if not os.path.exists(fp_fw):
            continue
        ds_fw = xr.open_dataset(fp_fw)
        end_area = float(ds_fw['area_m2'].values[-1]) / 1e6
        end_vol = float(ds_fw['volume_m3'].values[-1]) / 1e9
        ds_fw.close()

        # spin-up结果
        fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_SPINUP)
        ds_sp = xr.open_dataset(fp_sp)
        lia_init_area = float(ds_sp['area_m2'].values[0]) / 1e6
        spinup_end_area = float(ds_sp['area_m2'].values[-1]) / 1e6
        ds_sp.close()

        rgi_a = rgi_area_map.get(rgi_id, np.nan)
        cp = lia_calib_params[rgi_id]

        results.append({
            'uid': uid,
            'rgi_id': rgi_id,
            'calib_melt_f': cp['melt_f'],
            'orig_melt_f': original_params[rgi_id]['melt_f'],
            'mf_ratio': cp['melt_f'] / original_params[rgi_id]['melt_f'],
            'temp_bias': cp['temp_bias'],
            'lia_init_area': lia_init_area,
            'spinup_end_area': spinup_end_area,
            'spinup_chg_pct': (spinup_end_area - lia_init_area) / lia_init_area * 100,
            'forward_end_area': end_area,
            'rgi_area': rgi_a,
            'sim_rgi_ratio': end_area / rgi_a if rgi_a > 0 else np.nan,
        })
    except Exception as e:
        print("  提取失败 {}: {}".format(uid, e))

print("  提取成功: {} 条".format(len(results)))

# %% Cell 9 - 对比分析

print("\n" + "=" * 110)
print("[Step 5] 结果分析")
print("=" * 110)

df = pd.DataFrame(results)

print("\n{:<8} {:>8} {:>8} {:>8} {:>10} {:>10} {:>10} {:>8} {:>10}".format(
    "UID", "orig_mf", "cal_mf", "mf_rat", "LIA_init", "SP_end", "FW_end",
    "RGI", "sim/RGI"))
print("-" * 110)

for _, row in df.sort_values('sim_rgi_ratio').iterrows():
    print("{:<8} {:>8.2f} {:>8.2f} {:>8.2f} {:>10.2f} {:>10.2f} {:>10.2f} {:>8.2f} {:>10.3f}".format(
        row['uid'], row['orig_melt_f'], row['calib_melt_f'], row['mf_ratio'],
        row['lia_init_area'], row['spinup_end_area'], row['forward_end_area'],
        row['rgi_area'], row['sim_rgi_ratio']))

print("-" * 110)

ratios = df['sim_rgi_ratio'].dropna()
n_good = ((ratios >= 0.8) & (ratios <= 1.2)).sum()
print("\nsim/RGI: 均值={:.3f}  中位数={:.3f}  std={:.3f}".format(
    ratios.mean(), ratios.median(), ratios.std()))
print("[0.8-1.2]: {}/{} = {:.1f}%".format(
    n_good, len(ratios), n_good / len(ratios) * 100 if len(ratios) > 0 else 0))
print("spin-up面积变化均值: {:.1f}%".format(df['spinup_chg_pct'].mean()))
print("melt_f缩放比均值: {:.2f}  范围: [{:.2f}, {:.2f}]".format(
    df['mf_ratio'].mean(), df['mf_ratio'].min(), df['mf_ratio'].max()))

# %% Cell 10 - 可视化

import matplotlib.pyplot as plt

# 图1: 逐冰川 spinup+正演 时间序列
n_glaciers = len(spinup_ok)
ncols = 5
nrows = int(np.ceil(n_glaciers / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows))
axes = axes.flatten()

for idx, gdir in enumerate(spinup_ok):
    if idx >= len(axes):
        break
    ax = axes[idx]
    rgi_id = gdir.rgi_id
    uid = rgi_id.split('.')[-1]

    try:
        # spinup
        fp_sp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_SPINUP)
        ds_sp = xr.open_dataset(fp_sp)
        n_sp = len(ds_sp['area_m2'].values)
        years_sp = np.linspace(Y0 - SPINUP_YEARS, Y0, n_sp)
        area_sp = ds_sp['area_m2'].values / 1e6
        ds_sp.close()

        # forward
        fp_fw = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX_FORWARD)
        ds_fw = xr.open_dataset(fp_fw)
        n_fw = len(ds_fw['area_m2'].values)
        years_fw = np.arange(Y0, Y0 + n_fw)
        area_fw = ds_fw['area_m2'].values / 1e6
        ds_fw.close()

        ax.plot(years_sp, area_sp, color='gray', linewidth=1.5, alpha=0.6, label='spin-up')
        ax.plot(years_fw, area_fw, color='blue', linewidth=2.0, label='forward')
        ax.axvline(Y0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

        rgi_a = rgi_area_map.get(rgi_id, np.nan)
        ax.axhline(area_sp[0], color='green', linestyle=':', linewidth=1.5)
        ax.axhline(rgi_a, color='orange', linestyle='--', linewidth=1.8)

        row = df[df['rgi_id'] == rgi_id]
        if len(row) > 0:
            r = row.iloc[0]
            ax.set_title('{} | mf={:.1f}→{:.1f}\nratio={:.3f}'.format(
                uid, r['orig_melt_f'], r['calib_melt_f'], r['sim_rgi_ratio']),
                fontsize=9)
    except:
        ax.set_title('{} | FAILED'.format(uid), fontsize=9)

    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=6)

for i in range(n_glaciers, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('MB≈0 calibration on LIA geometry | Spin-up + Forward\n'
             'Green:=LIA init | Orange--=RGI observed',
             fontsize=13, fontweight='bold')
fig.supxlabel('Year', fontsize=12)
fig.supylabel('Area (km²)', fontsize=12)
plt.tight_layout()
plt.show()

# 图2: melt_f缩放比 vs sim/RGI ratio
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df['mf_ratio'], df['sim_rgi_ratio'], s=80, c='blue',
           edgecolors='black', linewidth=0.5, alpha=0.7)
for _, row in df.iterrows():
    ax.annotate(row['uid'], (row['mf_ratio'], row['sim_rgi_ratio']),
                fontsize=7, alpha=0.6, xytext=(3, 3), textcoords='offset points')
ax.axhline(1.0, color='red', linestyle='--', linewidth=1.5)
ax.fill_between(ax.get_xlim(), 0.8, 1.2, alpha=0.1, color='green')
ax.set_xlabel('melt_f ratio (calibrated / original)')
ax.set_ylabel('sim_2000 / RGI area')
ax.set_title('MB≈0 calibration: melt_f scaling vs forward result')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# %% Cell 11 - 保存

csv_path = os.path.join(SAVE_DIR, 'mb0_calib_results.csv')
df.to_csv(csv_path, index=False)
print("CSV: {}".format(csv_path))

pkl_path = os.path.join(SAVE_DIR, 'mb0_calib_full.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump({
        'lia_calib_params': lia_calib_params,
        'results': results,
        'config': {
            'TEST_N': TEST_N, 'Y0': Y0, 'HALFSIZE': HALFSIZE,
            'SPINUP_YEARS': SPINUP_YEARS, 'MELT_F_MIN': MELT_F_MIN,
            'MELT_F_MAX': MELT_F_MAX,
        },
    }, f)
print("Pickle: {}".format(pkl_path))

# 恢复原始参数
print("\n恢复原始参数...")
for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    if rgi_id not in original_params:
        continue
    calib_path = os.path.join(gdir.dir, 'mb_calib.json')
    with open(calib_path, 'r') as f:
        p = json.load(f)
    p['melt_f'] = original_params[rgi_id]['melt_f']
    p['temp_bias'] = original_params[rgi_id]['temp_bias']
    p['prcp_fac'] = original_params[rgi_id]['prcp_fac']
    with open(calib_path, 'w') as f:
        json.dump(p, f, indent=2)
print("恢复完成")

print("\n" + "=" * 70)
print("全部完成！")
print("=" * 70)

In [ ]:
"""
RGI冰川平衡态测试
==================
目的：用RGI冰川 + CRU原始校准参数 + 现代气候，跑constant_climate到稳态
看平衡态面积 vs RGI面积的比值

这是闭环测试0.86的进一步诊断：
- 如果平衡态面积也≈0.86×RGI → 纯粹是"瞬态参数→平衡态"的系统偏差
- 如果平衡态面积≈RGI → 说明0.86来自降温-退缩过程本身的额外偏差

不涉及任何参数修改、offset、降温，是最干净的基准测试。
"""

import json, os, gc
import numpy as np
import xarray as xr
import pandas as pd
from oggm import cfg, tasks
from oggm.core.massbalance import MonthlyTIModel
from oggm.core.flowline import FluxBasedModel
from oggm.workflow import execute_entity_task

# ============================================================
# 0. 配置
# ============================================================

class MyMBModel(MonthlyTIModel):
    """关闭校准参数检查"""
    def __init__(self, gdir, **kwargs):
        kwargs['check_calib_params'] = False
        super().__init__(gdir, **kwargs)

# 参数设置
SPINUP_YEARS = 500        # 足够长以达到稳态
Y0 = 1985                 # CRU气候参考期中心（1970-2000）
HALFSIZE = 15             # 1970-2000
CLIMATE_FILENAME = 'climate_historical'  # CRU气候文件


USE_MP = True
cfg.PARAMS['use_multiprocessing'] = USE_MP
cfg.PARAMS['continue_on_error'] = True
cfg.PARAMS['mp_processes'] = 15
# ============================================================
# 1. 选择测试冰川
# ============================================================
# 注意：gdirs_rgi 应该已经在你的notebook环境中定义好了
# 如果要测试子集，取消下面的注释：
TEST_N = 20
test_gdirs = gdirs_rgi[:TEST_N]

# 全量运行：
# test_gdirs = gdirs_rgi

print("=" * 60)
print("RGI冰川平衡态测试")
print("=" * 60)
print("冰川数量: {}".format(len(test_gdirs)))
print("气候参考期: {} ± {} (即{}-{})".format(Y0, HALFSIZE, Y0-HALFSIZE, Y0+HALFSIZE))
print("spin-up年数: {}".format(SPINUP_YEARS))
print("多进程: {}".format(USE_MP))
print("=" * 60)

# ============================================================
# 2. 记录原始RGI面积
# ============================================================
print("\n[Step 1] 记录RGI原始面积...")

rgi_areas = {}
for gdir in test_gdirs:
    rgi_areas[gdir.rgi_id] = gdir.rgi_area_m2 / 1e6  # km2

print("  已记录 {} 条冰川面积".format(len(rgi_areas)))
print("  面积范围: {:.2f} - {:.2f} km²".format(
    min(rgi_areas.values()), max(rgi_areas.values())))

# ============================================================
# 3. 跑constant_climate到稳态
# ============================================================
print("\n[Step 2] 用CRU原始参数跑constant_climate...")
print("  参考气候: CRU {}-{} 平均".format(Y0-HALFSIZE, Y0+HALFSIZE))
print("  参数: 不做任何修改，直接使用mb_calib.json中的原始校准参数")

SUFFIX = '_equilibrium_test'

try:
    execute_entity_task(
        tasks.run_constant_climate,
        test_gdirs,
        nyears=SPINUP_YEARS,
        y0=Y0,
        halfsize=HALFSIZE,
        climate_filename=CLIMATE_FILENAME,
        # 如果CRU气候有特定后缀，在这里添加：
        # climate_input_filesuffix='_CRU',
        mb_model_class=MyMBModel,
        evolution_model=FluxBasedModel,
        output_filesuffix=SUFFIX,
        store_model_geometry=True,
    )
    print("  constant_climate运行完成")
except Exception as e:
    print("  [WARN] 批量执行异常: {}".format(e))

# ============================================================
# 4. 提取结果并计算ratio
# ============================================================
print("\n[Step 3] 提取稳态面积...")

results = []
for gdir in test_gdirs:
    rgi_id = gdir.rgi_id
    rgi_area = rgi_areas[rgi_id]

    try:
        fp = gdir.get_filepath('model_diagnostics', filesuffix=SUFFIX)
        ds = xr.open_dataset(fp)

        area_init = float(ds.area_m2.values[0]) / 1e6
        area_final = float(ds.area_m2.values[-1]) / 1e6
        volume_init = float(ds.volume_m3.values[0]) / 1e9
        volume_final = float(ds.volume_m3.values[-1]) / 1e9

        # 稳态判断：最后50年面积变化率
        if len(ds.area_m2) > 50:
            area_last50 = ds.area_m2.values[-50:] / 1e6
            area_trend_pct = (area_last50[-1] - area_last50[0]) / area_last50[0] * 100
        else:
            area_trend_pct = np.nan

        ratio = area_final / rgi_area

        results.append({
            'rgi_id': rgi_id,
            'rgi_area_km2': rgi_area,
            'init_area_km2': area_init,
            'final_area_km2': area_final,
            'ratio_final_rgi': ratio,
            'area_change_pct': (area_final - area_init) / area_init * 100,
            'last50yr_trend_pct': area_trend_pct,
            'volume_init_km3': volume_init,
            'volume_final_km3': volume_final,
        })

        ds.close()
    except Exception as e:
        print("  [SKIP] {}: {}".format(rgi_id, str(e)[:60]))

df = pd.DataFrame(results)

# ============================================================
# 5. 输出结果
# ============================================================
print("\n" + "=" * 80)
print("结果汇总")
print("=" * 80)
print("成功: {} / {}".format(len(df), len(test_gdirs)))

if len(df) > 0:
    print("\n--- 平衡态面积 / RGI面积 ratio ---")
    print("  均值:   {:.4f}".format(df['ratio_final_rgi'].mean()))
    print("  中位数: {:.4f}".format(df['ratio_final_rgi'].median()))
    print("  标准差: {:.4f}".format(df['ratio_final_rgi'].std()))
    print("  最小值: {:.4f}".format(df['ratio_final_rgi'].min()))
    print("  最大值: {:.4f}".format(df['ratio_final_rgi'].max()))

    print("\n--- 面积变化（init→final） ---")
    print("  均值:   {:.2f}%".format(df['area_change_pct'].mean()))
    print("  中位数: {:.2f}%".format(df['area_change_pct'].median()))

    print("\n--- 最后50年面积趋势（判断是否稳态） ---")
    print("  均值:   {:.4f}%".format(df['last50yr_trend_pct'].mean()))
    print("  中位数: {:.4f}%".format(df['last50yr_trend_pct'].median()))
    print("  |趋势|>1% 的冰川: {}".format((df['last50yr_trend_pct'].abs() > 1).sum()))

    # 分组统计
    print("\n--- ratio分布 ---")
    bins = [0, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 2.0]
    labels = ['<0.7', '0.7-0.8', '0.8-0.85', '0.85-0.9', '0.9-0.95',
              '0.95-1.0', '1.0-1.05', '1.05-1.1', '>1.1']
    df['ratio_bin'] = pd.cut(df['ratio_final_rgi'], bins=bins, labels=labels)
    print(df['ratio_bin'].value_counts().sort_index().to_string())

    # 逐冰川结果
    print("\n--- 逐冰川结果 ---")
    print("{:<18} {:>10} {:>10} {:>10} {:>10} {:>12}".format(
        'RGI_ID', 'RGI_km2', 'Init_km2', 'Final_km2', 'Ratio', 'Last50yr_%'))
    print("-" * 80)
    for _, row in df.sort_values('ratio_final_rgi').iterrows():
        uid = row['rgi_id'].split('.')[-1]
        print("{:<18} {:>10.2f} {:>10.2f} {:>10.2f} {:>10.4f} {:>12.4f}".format(
            uid, row['rgi_area_km2'], row['init_area_km2'],
            row['final_area_km2'], row['ratio_final_rgi'], row['last50yr_trend_pct']))

    # 与闭环测试0.86的对比
    print("\n" + "=" * 60)
    print("关键对比")
    print("=" * 60)
    eq_ratio = df['ratio_final_rgi'].mean()
    print("本实验（无扰动平衡态）ratio均值: {:.4f}".format(eq_ratio))
    print("闭环测试（降温-退缩后）ratio均值: ~0.86")
    if abs(eq_ratio - 0.86) < 0.05:
        print("→ 两者接近！说明0.86主要来自参数瞬态性，")
        print("  可以作为系统偏差校正因子使用。")
    elif eq_ratio > 0.92:
        print("→ 无扰动平衡态接近RGI，说明0.86中有一部分")
        print("  来自降温-退缩过程的额外偏差（路径依赖效应）。")
    else:
        print("→ 无扰动平衡态也偏小，确认参数瞬态性是主因，")
        print("  但0.86中可能还叠加了路径依赖效应。")

    # 保存结果
    save_path = os.path.join(cfg.PATHS['working_dir'], 'rgi_equilibrium_results.csv')
    df.to_csv(save_path, index=False)
    print("\n结果已保存: {}".format(save_path))

print("\n实验完成。")